In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


HBAR = 0.6582119569  # meV*ps


def resolver_y_graficar(
    g_meV=1.0,
    Delta_meV=5.0,
    gamma_meV=0.1,
    kappa_meV=0.1,
    P_meV=15.0,
    t_initial=0.0,
    t_final=150.0,
    n_to_plot=None,
    xlim=None,
    ylim=(0, 0.1),
    Nmax=100,
    num_points=1501,
    figsize=(12, 7),
    mostrar_leyenda=True
):
    """
    Resuelve numéricamente las Ecs. (9), (10) y (11)
    y grafica los elementos diagonales de la matriz densidad.

    PARÁMETROS FÍSICOS
    ------------------
    g_meV:
        Acoplamiento excitón-fotón, en meV.

    Delta_meV:
        Detuning excitón-cavidad, en meV.

    gamma_meV:
        Tasa de emisión espontánea hacia modos leaky, en meV.

    kappa_meV:
        Tasa de pérdida de fotones de la cavidad, en meV.

    P_meV:
        Tasa de bombeo incoherente, en meV.


    PARÁMETROS NUMÉRICOS
    --------------------
    t_initial:
        Tiempo inicial de integración, en ps.

    t_final:
        Tiempo final de integración, en ps.

    n_to_plot:
        Lista de valores de n que se quieren graficar.
        Ejemplo:
            [0, 1, 2, 3]
        o:
            list(range(0, 36))

    xlim:
        Límites del eje x.
        Ejemplo:
            (0, 150)

        Si se deja como None, usa:
            (t_initial, t_final)

    ylim:
        Límites del eje y.
        Ejemplo:
            (0, 0.1)

    Nmax:
        Truncamiento del espacio de Fock.

    num_points:
        Número de puntos temporales guardados.

    figsize:
        Tamaño de la figura.

    mostrar_leyenda:
        True  -> muestra la leyenda
        False -> no muestra la leyenda


    DEVUELVE
    --------
    fig, ax, datos

    donde datos es un diccionario que contiene:
        t
        G
        X
        ReC
        ImC
        trace
        mean_photons
        p_n_final
    """


    # ========================================================
    # 1. CONVERSIÓN DE UNIDADES
    # ========================================================

    # De meV a ps^{-1}

    g = g_meV / HBAR
    Delta = Delta_meV / HBAR
    gamma = gamma_meV / HBAR
    kappa = kappa_meV / HBAR
    P = P_meV / HBAR


    # ========================================================
    # 2. VALORES DE n A GRAFICAR
    # ========================================================

    if n_to_plot is None:
        n_to_plot = list(range(0, 36))

    # Comprobamos que ningún n sea mayor que Nmax
    n_to_plot = [n for n in n_to_plot if 0 <= n <= Nmax]

    if len(n_to_plot) == 0:
        raise ValueError(
            "n_to_plot no contiene ningún valor válido "
            "entre 0 y Nmax."
        )


    # ========================================================
    # 3. TAMAÑO DE LOS ARRAYS
    # ========================================================

    nG = Nmax + 1
    nX = Nmax + 1
    nC = Nmax


    # ========================================================
    # 4. FUNCIÓN PARA SEPARAR EL VECTOR y
    # ========================================================

    def unpack(y):

        G = y[0:nG]

        X = y[nG:nG + nX]

        ReC = y[
            nG + nX:
            nG + nX + nC
        ]

        ImC = y[
            nG + nX + nC:
        ]

        return G, X, ReC, ImC


    # ========================================================
    # 5. SISTEMA DE ECUACIONES DIFERENCIALES
    # ========================================================

    def master_equations(t, y):

        G, X, ReC, ImC = unpack(y)

        dG = np.zeros_like(G)
        dX = np.zeros_like(X)

        dReC = np.zeros_like(ReC)
        dImC = np.zeros_like(ImC)


        # ====================================================
        # ECUACIÓN (9)
        # ====================================================

        for n in range(Nmax + 1):

            if n == 0:
                ImC_n = 0.0
            else:
                ImC_n = ImC[n - 1]


            if n < Nmax:
                G_next = G[n + 1]
            else:
                G_next = 0.0


            dG[n] = (

                -2.0 * g
                * np.sqrt(n)
                * ImC_n

                + gamma * X[n]

                - kappa * (
                    n * G[n]
                    - (n + 1) * G_next
                )

                - P * G[n]
            )


        # ====================================================
        # ECUACIÓN (10)
        # ====================================================

        for n in range(Nmax + 1):

            if n < Nmax:
                ImC_next = ImC[n]
            else:
                ImC_next = 0.0


            if n < Nmax:
                X_next = X[n + 1]
            else:
                X_next = 0.0


            dX[n] = (

                2.0 * g
                * np.sqrt(n + 1)
                * ImC_next

                - gamma * X[n]

                - kappa * (
                    n * X[n]
                    - (n + 1) * X_next
                )

                + P * G[n]
            )


        # ====================================================
        # ECUACIÓN (11)
        # ====================================================

        for n in range(1, Nmax + 1):

            j = n - 1


            Gamma_n = (
                gamma
                + kappa * (2 * n - 1)
                + P
            ) / 2.0


            if n < Nmax:

                ReC_next = ReC[j + 1]
                ImC_next = ImC[j + 1]

            else:

                ReC_next = 0.0
                ImC_next = 0.0


            # -----------------------------------------------
            # Parte real
            # -----------------------------------------------

            dReC[j] = (

                -Delta * ImC[j]

                - Gamma_n * ReC[j]

                + kappa
                * np.sqrt(n * (n + 1))
                * ReC_next
            )


            # -----------------------------------------------
            # Parte imaginaria
            # -----------------------------------------------

            dImC[j] = (

                g
                * np.sqrt(n)
                * (G[n] - X[n - 1])

                + Delta * ReC[j]

                - Gamma_n * ImC[j]

                + kappa
                * np.sqrt(n * (n + 1))
                * ImC_next
            )


        return np.concatenate(
            [dG, dX, dReC, dImC]
        )


    # ========================================================
    # 6. CONDICIONES INICIALES
    # ========================================================

    number_variables = (
        2 * (Nmax + 1)
        + 2 * Nmax
    )

    y0 = np.zeros(number_variables)

    # rho_G0,G0 = 1
    y0[0] = 1.0


    # ========================================================
    # 7. TIEMPOS
    # ========================================================

    t_eval = np.linspace(
        t_initial,
        t_final,
        num_points
    )


    # ========================================================
    # 8. RESOLVER SISTEMA
    # ========================================================

    solution = solve_ivp(

        master_equations,

        t_span=(
            t_initial,
            t_final
        ),

        y0=y0,

        t_eval=t_eval,

        method="BDF",

        rtol=1e-8,
        atol=1e-10
    )


    if not solution.success:

        raise RuntimeError(
            "La integración falló: "
            + solution.message
        )


    # ========================================================
    # 9. EXTRAER RESULTADOS
    # ========================================================

    t = solution.t


    G = solution.y[
        0:nG,
        :
    ]


    X = solution.y[
        nG:nG + nX,
        :
    ]


    ReC = solution.y[
        nG + nX:
        nG + nX + nC,
        :
    ]


    ImC = solution.y[
        nG + nX + nC:
        ,
        :
    ]


    # ========================================================
    # 10. CANTIDADES ÚTILES
    # ========================================================

    trace_rho = np.sum(
        G + X,
        axis=0
    )


    p_n_final = (
        G[:, -1]
        + X[:, -1]
    )


    n_values = np.arange(
        Nmax + 1
    )


    mean_photons = np.sum(

        n_values[:, None]
        * (G + X),

        axis=0
    )


    # ========================================================
    # 11. INFORMACIÓN EN PANTALLA
    # ========================================================

    print("\n====================================")
    print("PARÁMETROS DE LA SIMULACIÓN")
    print("====================================")

    print(f"g     = {g_meV} meV")
    print(f"Delta = {Delta_meV} meV")
    print(f"gamma = {gamma_meV} meV")
    print(f"kappa = {kappa_meV} meV")
    print(f"P     = {P_meV} meV")

    print()
    print(f"Tiempo = {t_initial} - {t_final} ps")
    print(f"Nmax   = {Nmax}")

    print()
    print(
        "Traza final =",
        trace_rho[-1]
    )

    print(
        "Número medio de fotones final =",
        mean_photons[-1]
    )

    print(
        f"Población en n={Nmax} =",
        p_n_final[Nmax]
    )


    # ========================================================
    # 12. GRÁFICA
    # ========================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )


    colors = plt.cm.viridis(
        np.linspace(
            0,
            1,
            len(n_to_plot)
        )
    )


    for color, n in zip(
        colors,
        n_to_plot
    ):


        # ----------------------------------------------------
        # Estado fundamental
        # Línea continua
        # ----------------------------------------------------

        ax.plot(

            t,
            G[n],

            color=color,
            linestyle="-",
            linewidth=1.5,

            label=fr"$\rho_{{G{n},G{n}}}$"
        )


        # ----------------------------------------------------
        # Estado excitónico
        # Línea discontinua
        # ----------------------------------------------------

        ax.plot(

            t,
            X[n],

            color=color,
            linestyle="--",
            linewidth=1.5,

            label=fr"$\rho_{{X{n},X{n}}}$"
        )


    # ========================================================
    # 13. LÍMITES DE LOS EJES
    # ========================================================

    if xlim is None:

        ax.set_xlim(
            t_initial,
            t_final
        )

    else:

        ax.set_xlim(
            xlim[0],
            xlim[1]
        )


    if ylim is not None:

        ax.set_ylim(
            ylim[0],
            ylim[1]
        )


    # ========================================================
    # 14. FORMATO
    # ========================================================

    ax.set_xlabel(
        "t (ps)"
    )

    ax.set_ylabel(
        r"$\rho_{in,in}$"
    )


    ax.set_title(

        rf"$g={g_meV}$ meV, "
        rf"$\Delta={Delta_meV}$ meV, "
        rf"$\gamma={gamma_meV}$ meV, "
        rf"$\kappa={kappa_meV}$ meV, "
        rf"$P={P_meV}$ meV"
    )


    if mostrar_leyenda:

        ax.legend(
            fontsize=7,
            ncol=3,
            bbox_to_anchor=(1.02, 1),
            loc="upper left"
        )


    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    plt.show()


    # ========================================================
    # 15. DEVOLVER RESULTADOS
    # ========================================================

    datos = {

        "t": t,

        "G": G,

        "X": X,

        "ReC": ReC,

        "ImC": ImC,

        "trace": trace_rho,

        "p_n_final": p_n_final,

        "mean_photons": mean_photons,

        "solution": solution
    }


    return fig, ax, datos



In [ ]:
fig, ax, datos = resolver_y_graficar(

    g_meV=1.0,
    Delta_meV=5.0,
    gamma_meV=0.1,
    kappa_meV=0.1,
    P_meV=15.0,

    t_initial=0,
    t_final=150,

    n_to_plot=list(range(0, 36)),

    xlim=(0, 150),
    ylim=(0, 0.10),

    Nmax=100
)

In [ ]:
fig, ax, datos = resolver_y_graficar(

    g_meV=1.0,
    Delta_meV=5.0,
    gamma_meV=0.1,
    kappa_meV=0.1,
    P_meV=15.0,

    t_initial=0,
    t_final=10,

    n_to_plot=list(range(0, 36)),

    xlim=(0, 10),
    ylim=(0, 1),

    Nmax=100
)

In [ ]:
fig, ax, datos = resolver_y_graficar(

    g_meV=1.0,
    Delta_meV=5.0,
    gamma_meV=0.1,
    kappa_meV=0.5,
    P_meV=1.0,

    t_initial=0,
    t_final=10,

    n_to_plot=list(range(0, 4, 1)),

    xlim=(0, 10),
    ylim=(0, 1.0),

    Nmax=100
)

In [ ]:
fig, ax, datos = resolver_y_graficar(

    g_meV=1.0,
    Delta_meV=5.0,
    gamma_meV=0.1,
    kappa_meV=0.5,
    P_meV=1.0,

    t_initial=0,
    t_final=10,

    n_to_plot=list(range(0, 4, 1)),

    xlim=(0, 10),
    ylim=(0, 0.005),

    Nmax=100
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from itertools import product


# ============================================================
# CONSTANTE
# ============================================================

HBAR = 0.6582119569  # meV*ps


# ============================================================
# FUNCIÓN PRINCIPAL
# ============================================================

def graficar_elemento_matriz(
    i,
    n_i,
    j,
    n_j,
    Delta_meV,
    g_meV,
    gamma_meV,
    kappa_meV,
    P_meV,
    t_initial=0.0,
    t_final=10.0,
    xlim=None,
    ylim=None,
    Nmax=100,
    num_points=1501,
    parte="ambas",
    combinar="pares",
    restaurar_fase=True,
    figsize=(11, 6)
):
    """
    Resuelve las Ecs. (9)-(11) y grafica un elemento de la
    matriz densidad.

    ----------------------------------------------------------
    ELEMENTO SOLICITADO
    ----------------------------------------------------------

    Se especifica como:

        (i, n_i, j, n_j)

    Ejemplo:

        ("X", 0, "G", 1)

    representa:

        rho_{X0,G1}


    Elementos disponibles:

        rho_{Gn,Gn}
        rho_{Xn,Xn}
        rho_{Gn,X(n-1)}
        rho_{X(n-1),Gn}


    ----------------------------------------------------------
    PARÁMETROS
    ----------------------------------------------------------

    Delta_meV : detuning [meV]
    g_meV     : acoplamiento excitón-fotón [meV]
    gamma_meV : emisión espontánea [meV]

    kappa_meV : número o lista de números [meV]
    P_meV     : número o lista de números [meV]


    ----------------------------------------------------------
    COMBINACIÓN DE kappa Y P
    ----------------------------------------------------------

    combinar="pares"

        kappa = [5, 0.1]
        P     = [1, 15]

    produce:

        (5, 1)
        (0.1, 15)


    combinar="producto"

    produce todas las combinaciones posibles.


    ----------------------------------------------------------
    PARTE A GRAFICAR
    ----------------------------------------------------------

    parte="real"
    parte="imag"
    parte="ambas"
    parte="abs"
    parte="fase"


    ----------------------------------------------------------
    RESTAURAR FASE
    ----------------------------------------------------------

    restaurar_fase=True

    Para coherencias, multiplica por la fase asociada al
    detuning:

        rho_GX -> exp(+i Delta t)
        rho_XG -> exp(-i Delta t)

    donde Delta ya fue convertido a ps^-1.


    ----------------------------------------------------------
    DEVUELVE
    ----------------------------------------------------------

    fig, ax, resultados
    """


    # ========================================================
    # 1. VALIDAR ÍNDICES
    # ========================================================

    i = i.upper()
    j = j.upper()

    if i not in ["G", "X"] or j not in ["G", "X"]:
        raise ValueError(
            "i y j solo pueden ser 'G' o 'X'."
        )

    if not (0 <= n_i <= Nmax):
        raise ValueError(
            "n_i debe estar entre 0 y Nmax."
        )

    if not (0 <= n_j <= Nmax):
        raise ValueError(
            "n_j debe estar entre 0 y Nmax."
        )


    # ========================================================
    # 2. IDENTIFICAR TIPO DE ELEMENTO
    # ========================================================

    diagonal_G = (
        i == "G"
        and j == "G"
        and n_i == n_j
    )

    diagonal_X = (
        i == "X"
        and j == "X"
        and n_i == n_j
    )

    # rho_{Gn,X(n-1)}
    coherencia_GX = (
        i == "G"
        and j == "X"
        and n_i == n_j + 1
    )

    # rho_{X(n-1),Gn}
    coherencia_XG = (
        i == "X"
        and j == "G"
        and n_j == n_i + 1
    )


    if not (
        diagonal_G
        or diagonal_X
        or coherencia_GX
        or coherencia_XG
    ):

        raise ValueError(
            "\nEl elemento solicitado no está incluido "
            "en las Ecs. (9)-(11).\n\n"
            "Elementos disponibles:\n"
            "rho_{Gn,Gn}\n"
            "rho_{Xn,Xn}\n"
            "rho_{Gn,X(n-1)}\n"
            "rho_{X(n-1),Gn}\n"
        )


    # ========================================================
    # 3. CONVERTIR PARÁMETROS A ps^-1
    # ========================================================

    g = g_meV / HBAR
    Delta = Delta_meV / HBAR
    gamma = gamma_meV / HBAR


    # ========================================================
    # 4. CONVERTIR kappa Y P A LISTAS
    # ========================================================

    def convertir_a_lista(valor):

        if np.isscalar(valor):
            return [float(valor)]

        return [float(x) for x in valor]


    kappas = convertir_a_lista(kappa_meV)
    Ps = convertir_a_lista(P_meV)


    # ========================================================
    # 5. CREAR LOS CASOS (kappa,P)
    # ========================================================

    if combinar == "pares":

        # Si kappa tiene un solo valor,
        # se usa para todos los P
        if len(kappas) == 1 and len(Ps) > 1:

            kappas = (
                kappas * len(Ps)
            )

        # Si P tiene un solo valor,
        # se usa para todos los kappa
        elif len(Ps) == 1 and len(kappas) > 1:

            Ps = (
                Ps * len(kappas)
            )

        elif len(kappas) != len(Ps):

            raise ValueError(
                "Para combinar='pares', kappa y P "
                "deben tener la misma longitud, salvo "
                "que uno de ellos sea un solo valor."
            )


        casos = list(
            zip(kappas, Ps)
        )


    elif combinar == "producto":

        casos = list(
            product(kappas, Ps)
        )


    else:

        raise ValueError(
            "combinar debe ser 'pares' o 'producto'."
        )


    # ========================================================
    # 6. DIMENSIONES DEL SISTEMA
    # ========================================================

    nG = Nmax + 1
    nX = Nmax + 1
    nC = Nmax

    number_variables = (
        2 * (Nmax + 1)
        + 2 * Nmax
    )


    # ========================================================
    # 7. SEPARAR VECTOR y
    # ========================================================

    def unpack(y):

        G = y[
            0:nG
        ]

        X = y[
            nG:
            nG + nX
        ]

        ReC = y[
            nG + nX:
            nG + nX + nC
        ]

        ImC = y[
            nG + nX + nC:
        ]

        return G, X, ReC, ImC


    # ========================================================
    # 8. RESOLVER UN CASO PARTICULAR
    # ========================================================

    def resolver(
        kappa_meV_caso,
        P_meV_caso
    ):

        # ----------------------------------------------------
        # Conversión a ps^-1
        # ----------------------------------------------------

        kappa = (
            kappa_meV_caso / HBAR
        )

        P = (
            P_meV_caso / HBAR
        )


        # ====================================================
        # SISTEMA DE ECUACIONES
        # ====================================================

        def master_equations(t, y):

            G, X, ReC, ImC = unpack(y)

            dG = np.zeros_like(G)
            dX = np.zeros_like(X)

            dReC = np.zeros_like(ReC)
            dImC = np.zeros_like(ImC)


            # =================================================
            # ECUACIÓN (9)
            # =================================================

            for n in range(
                Nmax + 1
            ):

                # C_n
                if n == 0:

                    ImC_n = 0.0

                else:

                    ImC_n = (
                        ImC[n - 1]
                    )


                # G_{n+1}
                if n < Nmax:

                    G_next = (
                        G[n + 1]
                    )

                else:

                    G_next = 0.0


                dG[n] = (

                    -2.0
                    * g
                    * np.sqrt(n)
                    * ImC_n

                    + gamma
                    * X[n]

                    - kappa
                    * (
                        n * G[n]
                        - (n + 1)
                        * G_next
                    )

                    - P
                    * G[n]
                )


            # =================================================
            # ECUACIÓN (10)
            # =================================================

            for n in range(
                Nmax + 1
            ):

                # C_{n+1}
                if n < Nmax:

                    ImC_next = (
                        ImC[n]
                    )

                else:

                    ImC_next = 0.0


                # X_{n+1}
                if n < Nmax:

                    X_next = (
                        X[n + 1]
                    )

                else:

                    X_next = 0.0


                dX[n] = (

                    2.0
                    * g
                    * np.sqrt(n + 1)
                    * ImC_next

                    - gamma
                    * X[n]

                    - kappa
                    * (
                        n * X[n]
                        - (n + 1)
                        * X_next
                    )

                    + P
                    * G[n]
                )


            # =================================================
            # ECUACIÓN (11)
            # =================================================

            for n in range(
                1,
                Nmax + 1
            ):

                index = n - 1


                Gamma_n = (

                    gamma

                    + kappa
                    * (2 * n - 1)

                    + P

                ) / 2.0


                # C_{n+1}
                if n < Nmax:

                    ReC_next = (
                        ReC[index + 1]
                    )

                    ImC_next = (
                        ImC[index + 1]
                    )

                else:

                    ReC_next = 0.0
                    ImC_next = 0.0


                # ---------------------------------------------
                # PARTE REAL
                # ---------------------------------------------

                dReC[index] = (

                    -Delta
                    * ImC[index]

                    - Gamma_n
                    * ReC[index]

                    + kappa
                    * np.sqrt(
                        n * (n + 1)
                    )
                    * ReC_next
                )


                # ---------------------------------------------
                # PARTE IMAGINARIA
                # ---------------------------------------------

                dImC[index] = (

                    g
                    * np.sqrt(n)
                    * (
                        G[n]
                        - X[n - 1]
                    )

                    + Delta
                    * ReC[index]

                    - Gamma_n
                    * ImC[index]

                    + kappa
                    * np.sqrt(
                        n * (n + 1)
                    )
                    * ImC_next
                )


            return np.concatenate(
                [
                    dG,
                    dX,
                    dReC,
                    dImC
                ]
            )


        # ====================================================
        # CONDICIONES INICIALES
        # ====================================================

        y0 = np.zeros(
            number_variables
        )

        # rho_G0,G0 = 1
        y0[0] = 1.0


        # ====================================================
        # TIEMPOS
        # ====================================================

        t_eval = np.linspace(
            t_initial,
            t_final,
            num_points
        )


        # ====================================================
        # INTEGRACIÓN NUMÉRICA
        # ====================================================

        solution = solve_ivp(

            master_equations,

            t_span=(
                t_initial,
                t_final
            ),

            y0=y0,

            t_eval=t_eval,

            method="BDF",

            rtol=1e-8,

            atol=1e-10
        )


        if not solution.success:

            raise RuntimeError(
                "La integración falló: "
                + solution.message
            )


        # ====================================================
        # EXTRAER RESULTADOS
        # ====================================================

        t = solution.t


        G = solution.y[
            0:nG,
            :
        ]


        X = solution.y[
            nG:
            nG + nX,
            :
        ]


        ReC = solution.y[
            nG + nX:
            nG + nX + nC,
            :
        ]


        ImC = solution.y[
            nG + nX + nC:
            ,
            :
        ]


        # ====================================================
        # SELECCIONAR ELEMENTO DE MATRIZ
        # ====================================================

        # ----------------------------------------------------
        # DIAGONAL G
        # ----------------------------------------------------

        if diagonal_G:

            rho = (
                G[n_i]
                .astype(complex)
            )


        # ----------------------------------------------------
        # DIAGONAL X
        # ----------------------------------------------------

        elif diagonal_X:

            rho = (
                X[n_i]
                .astype(complex)
            )


        # ----------------------------------------------------
        # rho_{Gn,X(n-1)}
        # ----------------------------------------------------

        elif coherencia_GX:

            index = (
                n_i - 1
            )


            rho_tilde = (

                ReC[index]

                + 1j
                * ImC[index]
            )


            if restaurar_fase:

                rho = (

                    rho_tilde

                    * np.exp(
                        +1j
                        * Delta
                        * t
                    )
                )

            else:

                rho = rho_tilde


        # ----------------------------------------------------
        # rho_{X(n-1),Gn}
        # ----------------------------------------------------

        elif coherencia_XG:

            index = (
                n_j - 1
            )


            # conjugado hermítico

            rho_tilde = (

                ReC[index]

                - 1j
                * ImC[index]
            )


            if restaurar_fase:

                rho = (

                    rho_tilde

                    * np.exp(
                        -1j
                        * Delta
                        * t
                    )
                )

            else:

                rho = rho_tilde


        return (
            t,
            rho,
            G,
            X,
            ReC,
            ImC
        )


    # ========================================================
    # 9. CREAR FIGURA
    # ========================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )


    # Un color para cada par (kappa,P)

    colors = plt.cm.tab10(
        np.linspace(
            0,
            1,
            len(casos)
        )
    )


    resultados = []


    # ========================================================
    # 10. RESOLVER Y GRAFICAR CADA CASO
    # ========================================================

    for color, (
        kappa_caso,
        P_caso
    ) in zip(
        colors,
        casos
    ):


        (
            t,
            rho,
            G,
            X,
            ReC,
            ImC

        ) = resolver(
            kappa_caso,
            P_caso
        )


        # ----------------------------------------------------
        # PARTE REAL
        # ----------------------------------------------------

        if parte == "real":

            ax.plot(

                t,
                np.real(rho),

                color=color,

                linestyle="-",

                linewidth=1.7,

                label=(
                    rf"$\kappa={kappa_caso}$, "
                    rf"$P={P_caso}$ (Re)"
                )
            )


        # ----------------------------------------------------
        # PARTE IMAGINARIA
        # ----------------------------------------------------

        elif parte == "imag":

            ax.plot(

                t,
                np.imag(rho),

                color=color,

                linestyle="-",

                linewidth=1.7,

                label=(
                    rf"$\kappa={kappa_caso}$, "
                    rf"$P={P_caso}$ (Im)"
                )
            )


        # ----------------------------------------------------
        # AMBAS
        # ----------------------------------------------------

        elif parte == "ambas":

            # Real: continua
            ax.plot(

                t,
                np.real(rho),

                color=color,

                linestyle="-",

                linewidth=1.7,

                label=(
                    rf"$\kappa={kappa_caso}$, "
                    rf"$P={P_caso}$ (Re)"
                )
            )


            # Imaginaria: discontinua
            ax.plot(

                t,
                np.imag(rho),

                color=color,

                linestyle="--",

                linewidth=1.7,

                label=(
                    rf"$\kappa={kappa_caso}$, "
                    rf"$P={P_caso}$ (Im)"
                )
            )


        # ----------------------------------------------------
        # VALOR ABSOLUTO
        # ----------------------------------------------------

        elif parte == "abs":

            ax.plot(

                t,
                np.abs(rho),

                color=color,

                linewidth=1.7,

                label=(
                    rf"$\kappa={kappa_caso}$, "
                    rf"$P={P_caso}$"
                )
            )


        # ----------------------------------------------------
        # FASE
        # ----------------------------------------------------

        elif parte == "fase":

            ax.plot(

                t,
                np.angle(rho),

                color=color,

                linewidth=1.7,

                label=(
                    rf"$\kappa={kappa_caso}$, "
                    rf"$P={P_caso}$"
                )
            )


        else:

            raise ValueError(
                "parte debe ser:"
                " 'real', 'imag', 'ambas',"
                " 'abs' o 'fase'."
            )


        # Guardar resultados

        resultados.append({

            "kappa_meV":
                kappa_caso,

            "P_meV":
                P_caso,

            "t":
                t,

            "rho":
                rho,

            "real":
                np.real(rho),

            "imag":
                np.imag(rho),

            "abs":
                np.abs(rho),

            "G":
                G,

            "X":
                X,

            "ReC":
                ReC,

            "ImC":
                ImC
        })


    # ========================================================
    # 11. LÍMITES
    # ========================================================

    if xlim is None:

        ax.set_xlim(
            t_initial,
            t_final
        )

    else:

        ax.set_xlim(
            xlim
        )


    if ylim is not None:

        ax.set_ylim(
            ylim
        )


    # ========================================================
    # 12. ETIQUETAS
    # ========================================================

    nombre_elemento = (
        rf"\rho_{{{i}{n_i},{j}{n_j}}}"
    )


    ax.set_xlabel(
        "t (ps)"
    )


    if parte == "real":

        ax.set_ylabel(
            rf"$\mathrm{{Re}}"
            rf"[{nombre_elemento}]$"
        )

    elif parte == "imag":

        ax.set_ylabel(
            rf"$\mathrm{{Im}}"
            rf"[{nombre_elemento}]$"
        )

    elif parte == "abs":

        ax.set_ylabel(
            rf"$|{nombre_elemento}|$"
        )

    elif parte == "fase":

        ax.set_ylabel(
            rf"$\arg({nombre_elemento})$"
        )

    else:

        ax.set_ylabel(
            rf"${nombre_elemento}$"
        )


    # ========================================================
    # 13. TÍTULO
    # ========================================================

    ax.set_title(

        rf"${nombre_elemento}$"
        "\n"

        rf"$g={g_meV}$ meV, "

        rf"$\Delta={Delta_meV}$ meV, "

        rf"$\gamma={gamma_meV}$ meV"
    )


    # ========================================================
    # 14. FORMATO
    # ========================================================

    ax.legend()

    ax.grid(
        alpha=0.3
    )

    fig.tight_layout()

    plt.show()


    # ========================================================
    # 15. DEVOLVER RESULTADOS
    # ========================================================

    return (
        fig,
        ax,
        resultados
    )

fig, ax, resultados = graficar_elemento_matriz(

    # Elemento rho_X0,G1
    "X", 0,
    "G", 1,

    # Parámetros comunes
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,

    # Dos casos del paper
    kappa_meV=[5, 0.1],
    P_meV=[1, 15],

    # Combinar:
    # (kappa=5, P=1)
    # (kappa=0.1, P=15)
    combinar="pares",

    # Tiempo calculado
    t_initial=0,
    t_final=10,

    # Límites de gráfica
    xlim=(0, 10),
    ylim=(-0.15, 0.15),

    # Graficar Re e Im simultáneamente
    parte="ambas",

    # Restaurar fase temporal del detuning
    restaurar_fase=True,

    # Truncamiento
    Nmax=100,

    # Resolución temporal
    num_points=2001
)

In [ ]:
fig, ax, resultados = graficar_elemento_matriz(

    # Elemento rho_X0,G1
    "G", 1,
    "X", 0,


    # Parámetros comunes
    Delta_meV=0,
    g_meV=1,
    gamma_meV=0.1,

    # Dos casos del paper
    kappa_meV=[5, 0.1, 5],
    P_meV=[1, 15, 15],

    # Combinar:
    # (kappa=5, P=1)
    # (kappa=0.1, P=15)
    combinar="pares",

    # Tiempo calculado
    t_initial=0,
    t_final=25,

    # Límites de gráfica
    xlim=(0, 25),
    ylim=(-0.2, 0.0),

    # Graficar Re e Im simultáneamente
    parte="ambas",

    # Restaurar fase temporal del detuning
    restaurar_fase=True,

    # Truncamiento
    Nmax=100,

    # Resolución temporal
    num_points=2001
)

In [ ]:
fig, ax, resultados = graficar_elemento_matriz(

    # Elemento rho_X0,G1
    "G", 1,
    "X", 0,


    # Parámetros comunes
    Delta_meV=0,
    g_meV=1,
    gamma_meV=0.1,

    # Dos casos del paper
    kappa_meV=[5, 0.1, 5],
    P_meV=[1, 15, 15],

    # Combinar:
    # (kappa=5, P=1)
    # (kappa=0.1, P=15)
    combinar="pares",

    # Tiempo calculado
    t_initial=0,
    t_final=1.5,

    # Límites de gráfica
    xlim=(0, 1.5),
    ylim=(-0.2, 0.0),

    # Graficar Re e Im simultáneamente
    parte="ambas",

    # Restaurar fase temporal del detuning
    restaurar_fase=True,

    # Truncamiento
    Nmax=100,

    # Resolución temporal
    num_points=2001
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


# ============================================================
# CONSTANTE
# ============================================================

HBAR = 0.6582119569  # meV*ps


# ============================================================
# MAPA DE COLOR DEL NÚMERO PROMEDIO DE FOTONES
# ============================================================

def mapa_Nph(
    Delta_meV,
    g_meV,
    gamma_meV,
    xlim,
    ylim,
    Nmax=100,
    n_kappa=40,
    n_P=40,
    figsize=(9, 7),
    cmap="viridis",
    vmin=None,
    vmax=None,
    contours=None,
    mostrar_progreso=True
):
    """
    Calcula y grafica el número promedio estacionario
    de fotones:

        Nph(P, kappa)

    resolviendo directamente el sistema estacionario

        A y = 0

    junto con

        Tr(rho) = 1.


    ============================================================
    PARÁMETROS FÍSICOS
    ============================================================

    Delta_meV:
        Detuning Delta, en meV.

    g_meV:
        Acoplamiento excitón-fotón, en meV.

    gamma_meV:
        Tasa de emisión espontánea hacia modos leaky,
        en meV.


    ============================================================
    RANGOS DEL MAPA
    ============================================================

    xlim:
        Rango de kappa, en meV.

        Ejemplo:
            xlim=(0.01, 5)

    ylim:
        Rango de P, en meV.

        Ejemplo:
            ylim=(0.01, 20)


    ============================================================
    PARÁMETROS NUMÉRICOS
    ============================================================

    Nmax:
        Truncamiento del espacio de Fock.

    n_kappa:
        Número de puntos en el eje kappa.

    n_P:
        Número de puntos en el eje P.


    ============================================================
    GRÁFICA
    ============================================================

    cmap:
        Mapa de colores.

    vmin, vmax:
        Límites opcionales de la escala de colores.

    contours:
        Si es None, no dibuja contornos.

        Si es un entero, por ejemplo:
            contours=10

        dibuja ese número de curvas de nivel.


    ============================================================
    DEVUELVE
    ============================================================

    fig, ax, datos

    donde datos contiene:

        datos["kappa"]
        datos["P"]
        datos["Nph"]
        datos["border_population"]

    """


    # ========================================================
    # 1. CONVERSIÓN DE UNIDADES
    # ========================================================

    # Trabajamos internamente con ps^{-1}.
    #
    # Para un estado estacionario esta conversión común
    # realmente no cambia la solución, pero mantiene las
    # unidades consistentes con nuestro código anterior.

    Delta = Delta_meV / HBAR
    g = g_meV / HBAR
    gamma = gamma_meV / HBAR


    # ========================================================
    # 2. MALLA DE PARÁMETROS
    # ========================================================

    kappa_values_meV = np.linspace(
        xlim[0],
        xlim[1],
        n_kappa
    )

    P_values_meV = np.linspace(
        ylim[0],
        ylim[1],
        n_P
    )


    # Matriz donde guardaremos Nph(P, kappa)

    Nph_map = np.zeros(
        (n_P, n_kappa)
    )


    # También guardaremos la población en el borde n=Nmax.
    #
    # Esto sirve para comprobar si Nmax es suficientemente
    # grande.

    border_population = np.zeros(
        (n_P, n_kappa)
    )


    # ========================================================
    # 3. DIMENSIÓN DEL VECTOR DE VARIABLES
    # ========================================================

    # Orden del vector:
    #
    # [G_0 ... G_N,
    #  X_0 ... X_N,
    #  ReC_1 ... ReC_N,
    #  ImC_1 ... ImC_N]
    #
    # Total:
    #
    # 2(N+1) + 2N = 4N+2

    nG = Nmax + 1
    nX = Nmax + 1
    nC = Nmax

    dim = (
        2 * (Nmax + 1)
        + 2 * Nmax
    )


    # ========================================================
    # 4. ÍNDICES DENTRO DEL VECTOR
    # ========================================================

    def idx_G(n):

        return n


    def idx_X(n):

        return nG + n


    def idx_ReC(n):
        """
        n = 1,...,Nmax
        """

        return (
            nG
            + nX
            + (n - 1)
        )


    def idx_ImC(n):
        """
        n = 1,...,Nmax
        """

        return (
            nG
            + nX
            + nC
            + (n - 1)
        )


    # ========================================================
    # 5. FUNCIÓN QUE RESUELVE UN SOLO (P, kappa)
    # ========================================================

    def resolver_estado_estacionario(
        kappa_meV,
        P_meV
    ):

        # ----------------------------------------------------
        # Convertimos a ps^-1
        # ----------------------------------------------------

        kappa = (
            kappa_meV / HBAR
        )

        P = (
            P_meV / HBAR
        )


        # ----------------------------------------------------
        # Construimos la matriz A
        #
        # d y/dt = A y
        #
        # En régimen estacionario:
        #
        # A y = 0
        # ----------------------------------------------------

        A = lil_matrix(
            (dim, dim),
            dtype=float
        )


        # ====================================================
        # ECUACIÓN (9)
        #
        # dG_n/dt =
        #
        # -2 g sqrt(n) ImC_n
        # + gamma X_n
        # - kappa n G_n
        # + kappa(n+1)G_{n+1}
        # - P G_n
        # ====================================================

        for n in range(
            Nmax + 1
        ):

            row = idx_G(n)


            # G_n

            A[
                row,
                idx_G(n)
            ] += (
                -kappa * n
                - P
            )


            # X_n

            A[
                row,
                idx_X(n)
            ] += gamma


            # Im(C_n)

            if n >= 1:

                A[
                    row,
                    idx_ImC(n)
                ] += (
                    -2.0
                    * g
                    * np.sqrt(n)
                )


            # G_{n+1}

            if n < Nmax:

                A[
                    row,
                    idx_G(n + 1)
                ] += (
                    kappa
                    * (n + 1)
                )


        # ====================================================
        # ECUACIÓN (10)
        #
        # dX_n/dt =
        #
        # +2 g sqrt(n+1) ImC_{n+1}
        # - gamma X_n
        # - kappa n X_n
        # + kappa(n+1)X_{n+1}
        # + P G_n
        # ====================================================

        for n in range(
            Nmax + 1
        ):

            row = idx_X(n)


            # G_n

            A[
                row,
                idx_G(n)
            ] += P


            # X_n

            A[
                row,
                idx_X(n)
            ] += (
                -gamma
                - kappa * n
            )


            # Im(C_{n+1})

            if n < Nmax:

                A[
                    row,
                    idx_ImC(n + 1)
                ] += (
                    2.0
                    * g
                    * np.sqrt(n + 1)
                )


            # X_{n+1}

            if n < Nmax:

                A[
                    row,
                    idx_X(n + 1)
                ] += (
                    kappa
                    * (n + 1)
                )


        # ====================================================
        # ECUACIÓN (11)
        #
        # Separamos en parte real e imaginaria.
        # ====================================================

        for n in range(
            1,
            Nmax + 1
        ):


            Gamma_n = (

                gamma

                + kappa
                * (2 * n - 1)

                + P

            ) / 2.0


            # =================================================
            # PARTE REAL
            #
            # d ReC_n/dt =
            #
            # - Delta ImC_n
            # - Gamma_n ReC_n
            # + kappa sqrt[n(n+1)] ReC_{n+1}
            # =================================================

            row_Re = idx_ReC(n)


            A[
                row_Re,
                idx_ReC(n)
            ] += (
                -Gamma_n
            )


            A[
                row_Re,
                idx_ImC(n)
            ] += (
                -Delta
            )


            if n < Nmax:

                A[
                    row_Re,
                    idx_ReC(n + 1)
                ] += (

                    kappa

                    * np.sqrt(
                        n * (n + 1)
                    )
                )


            # =================================================
            # PARTE IMAGINARIA
            #
            # d ImC_n/dt =
            #
            # g sqrt(n)(G_n-X_{n-1})
            #
            # + Delta ReC_n
            #
            # - Gamma_n ImC_n
            #
            # + kappa sqrt[n(n+1)] ImC_{n+1}
            # =================================================

            row_Im = idx_ImC(n)


            # G_n

            A[
                row_Im,
                idx_G(n)
            ] += (
                g
                * np.sqrt(n)
            )


            # X_{n-1}

            A[
                row_Im,
                idx_X(n - 1)
            ] += (
                -g
                * np.sqrt(n)
            )


            # ReC_n

            A[
                row_Im,
                idx_ReC(n)
            ] += Delta


            # ImC_n

            A[
                row_Im,
                idx_ImC(n)
            ] += (
                -Gamma_n
            )


            # ImC_{n+1}

            if n < Nmax:

                A[
                    row_Im,
                    idx_ImC(n + 1)
                ] += (

                    kappa

                    * np.sqrt(
                        n * (n + 1)
                    )
                )


        # ====================================================
        # 6. NORMALIZACIÓN
        #
        # A y = 0 tiene solución no trivial porque A es
        # singular.
        #
        # Reemplazamos una ecuación por:
        #
        # sum_n (G_n + X_n) = 1
        # ====================================================

        b = np.zeros(dim)


        normalization_row = 0


        # Borramos completamente esa fila

        A[
            normalization_row,
            :
        ] = 0.0


        # Insertamos la condición de normalización

        for n in range(
            Nmax + 1
        ):

            A[
                normalization_row,
                idx_G(n)
            ] = 1.0

            A[
                normalization_row,
                idx_X(n)
            ] = 1.0


        b[
            normalization_row
        ] = 1.0


        # ====================================================
        # 7. RESOLVER EL SISTEMA LINEAL
        # ====================================================

        A = A.tocsr()


        y_ss = spsolve(
            A,
            b
        )


        if not np.all(
            np.isfinite(y_ss)
        ):

            return (
                np.nan,
                np.nan
            )


        # ====================================================
        # 8. EXTRAER POBLACIONES
        # ====================================================

        G = np.array([
            y_ss[idx_G(n)]
            for n in range(Nmax + 1)
        ])


        X = np.array([
            y_ss[idx_X(n)]
            for n in range(Nmax + 1)
        ])


        # ====================================================
        # 9. NÚMERO MEDIO DE FOTONES
        # ====================================================

        n_values = np.arange(
            Nmax + 1
        )


        Nph = np.sum(

            n_values

            * (
                G + X
            )
        )


        # ====================================================
        # 10. POBLACIÓN EN EL BORDE
        # ====================================================

        p_border = (

            G[Nmax]

            + X[Nmax]
        )


        return (
            Nph,
            p_border
        )


    # ========================================================
    # 11. RECORRER TODO EL MAPA
    # ========================================================

    for iP, P_meV in enumerate(
        P_values_meV
    ):


        if mostrar_progreso:

            print(
                f"Fila {iP + 1}/{n_P} "
                f"- P = {P_meV:.4g} meV"
            )


        for ik, kappa_meV in enumerate(
            kappa_values_meV
        ):


            (
                Nph,
                p_border

            ) = resolver_estado_estacionario(

                kappa_meV,
                P_meV
            )


            Nph_map[
                iP,
                ik
            ] = Nph


            border_population[
                iP,
                ik
            ] = p_border


    # ========================================================
    # 12. CREAR MALLA PARA MATPLOTLIB
    # ========================================================

    KAPPA, PP = np.meshgrid(

        kappa_values_meV,

        P_values_meV
    )


    # ========================================================
    # 13. GRÁFICA
    # ========================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )


    color_map = ax.pcolormesh(

        KAPPA,
        PP,
        Nph_map,

        shading="auto",

        cmap=cmap,

        vmin=vmin,
        vmax=vmax
    )


    # --------------------------------------------------------
    # Barra de color
    # --------------------------------------------------------

    cbar = fig.colorbar(

        color_map,

        ax=ax
    )


    cbar.set_label(
        r"$N_{\mathrm{ph}}$"
    )


    # --------------------------------------------------------
    # Contornos opcionales
    # --------------------------------------------------------

    if contours is not None:

        contour_lines = ax.contour(

            KAPPA,
            PP,
            Nph_map,

            levels=contours,
            colors="red",

            linewidths=1.0
        )


        ax.clabel(

            contour_lines,

            inline=True,

            fontsize=8
        )


    # --------------------------------------------------------
    # Ejes
    # --------------------------------------------------------

    ax.set_xlim(
        xlim
    )

    ax.set_ylim(
        ylim
    )


    ax.set_xlabel(
        r"$\kappa$ (meV)"
    )

    ax.set_ylabel(
        r"$P$ (meV)"
    )


    # --------------------------------------------------------
    # Título
    # --------------------------------------------------------

    ax.set_title(

        r"Número promedio de fotones"
        "\n"

        rf"$g={g_meV}$ meV, "
        rf"$\Delta={Delta_meV}$ meV, "
        rf"$\gamma={gamma_meV}$ meV"
    )


    fig.tight_layout()

    plt.show()


    # ========================================================
    # 14. COMPROBACIÓN DEL TRUNCAMIENTO
    # ========================================================

    max_border = np.nanmax(
        np.abs(border_population)
    )


    print(
        "\nMáxima población encontrada "
        f"en n = {Nmax}: {max_border:.3e}"
    )


    if max_border > 1e-5:

        print(
            "ADVERTENCIA: la población en el borde "
            "no es completamente despreciable.\n"
            "Sería recomendable aumentar Nmax."
        )


    # ========================================================
    # 15. DEVOLVER DATOS
    # ========================================================

    datos = {

        "kappa":
            kappa_values_meV,

        "P":
            P_values_meV,

        "Nph":
            Nph_map,

        "border_population":
            border_population,

        "KAPPA_mesh":
            KAPPA,

        "P_mesh":
            PP
    }


    return (
        fig,
        ax,
        datos
    )

In [ ]:
fig, ax, datos = mapa_Nph(

    # Parámetros físicos
    Delta_meV=0,
    g_meV=1,
    gamma_meV=0.1,

    # Eje x: kappa
    xlim=(0.1, 0.5),

    # Eje y: P
    ylim=(0.05, 50),

    # Truncamiento
    Nmax=100,

    # Resolución del mapa
    n_kappa=100,
    n_P=100,

    # Curvas de nivel
    contours=10
)

In [ ]:
fig, ax, datos = mapa_Nph(

    # Parámetros físicos
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,

    # Eje x: kappa
    xlim=(0.1, 0.5),

    # Eje y: P
    ylim=(0.05, 50),

    # Truncamiento
    Nmax=100,

    # Resolución del mapa
    n_kappa=100,
    n_P=100,

    # Curvas de nivel
    contours=10
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


# ============================================================
# CONSTANTE
# ============================================================

HBAR = 0.6582119569  # meV*ps


# ============================================================
# MAPA DE COLOR DE g^(2)(0)
# ============================================================

def mapa_g2(
    Delta_meV,
    g_meV,
    gamma_meV,
    xlim,
    ylim,
    Nmax=100,
    n_kappa=40,
    n_P=40,
    figsize=(9, 7),
    cmap="viridis",
    vmin=0,
    vmax=2,
    contours=None,
    mostrar_progreso=True
):
    """
    Calcula y grafica:

                g^(2)(0)(P, kappa)

    en el estado estacionario.

    Se utiliza:

        g2 =
        sum_n n(n-1)[G_n + X_n]
        --------------------------------
        [sum_n n(G_n + X_n)]^2


    ============================================================
    PARÁMETROS FÍSICOS
    ============================================================

    Delta_meV:
        Detuning, en meV.

    g_meV:
        Acoplamiento excitón-fotón, en meV.

    gamma_meV:
        Emisión espontánea hacia modos leaky, en meV.


    ============================================================
    RANGOS
    ============================================================

    xlim:
        Rango de kappa, en meV.

        Ejemplo:
            xlim=(0.1, 2.5)

    ylim:
        Rango de P, en meV.

        Ejemplo:
            ylim=(0.1, 50)


    ============================================================
    PARÁMETROS NUMÉRICOS
    ============================================================

    Nmax:
        Truncamiento del espacio de Fock.

    n_kappa:
        Número de puntos en kappa.

    n_P:
        Número de puntos en P.


    ============================================================
    GRÁFICA
    ============================================================

    cmap:
        Mapa de colores.

    vmin, vmax:
        Límites de la escala de g2.

        Para comparar con las figuras del paper,
        vmin=0 y vmax=2 son convenientes.

    contours:
        None -> sin contornos.

        Un entero, por ejemplo:
            contours=10

        dibuja curvas de nivel.


    ============================================================
    DEVUELVE
    ============================================================

    fig, ax, datos

    datos contiene:

        datos["kappa"]
        datos["P"]
        datos["g2"]
        datos["Nph"]
        datos["border_population"]
    """


    # ========================================================
    # 1. CONVERSIÓN DE UNIDADES
    # ========================================================

    Delta = Delta_meV / HBAR
    g = g_meV / HBAR
    gamma = gamma_meV / HBAR


    # ========================================================
    # 2. MALLA DE PARÁMETROS
    # ========================================================

    kappa_values_meV = np.linspace(
        xlim[0],
        xlim[1],
        n_kappa
    )

    P_values_meV = np.linspace(
        ylim[0],
        ylim[1],
        n_P
    )


    # --------------------------------------------------------
    # Matrices donde guardamos resultados
    # --------------------------------------------------------

    g2_map = np.full(
        (n_P, n_kappa),
        np.nan
    )

    Nph_map = np.zeros(
        (n_P, n_kappa)
    )

    border_population = np.zeros(
        (n_P, n_kappa)
    )


    # ========================================================
    # 3. DIMENSIONES
    # ========================================================

    nG = Nmax + 1
    nX = Nmax + 1
    nC = Nmax

    dim = (
        2 * (Nmax + 1)
        + 2 * Nmax
    )


    # ========================================================
    # 4. ÍNDICES DEL VECTOR DE VARIABLES
    # ========================================================

    def idx_G(n):
        return n


    def idx_X(n):
        return nG + n


    def idx_ReC(n):
        """
        n = 1,...,Nmax
        """
        return (
            nG
            + nX
            + (n - 1)
        )


    def idx_ImC(n):
        """
        n = 1,...,Nmax
        """
        return (
            nG
            + nX
            + nC
            + (n - 1)
        )


    # ========================================================
    # 5. RESOLVER UN PUNTO (kappa,P)
    # ========================================================

    def resolver_estado_estacionario(
        kappa_meV,
        P_meV
    ):

        # ----------------------------------------------------
        # Conversión a ps^-1
        # ----------------------------------------------------

        kappa = kappa_meV / HBAR
        P = P_meV / HBAR


        # ----------------------------------------------------
        # Matriz A:
        #
        # dy/dt = A y
        # ----------------------------------------------------

        A = lil_matrix(
            (dim, dim),
            dtype=float
        )


        # ====================================================
        # ECUACIÓN (9)
        #
        # dG_n/dt
        # ====================================================

        for n in range(Nmax + 1):

            row = idx_G(n)


            # G_n

            A[
                row,
                idx_G(n)
            ] += (
                -kappa * n
                - P
            )


            # X_n

            A[
                row,
                idx_X(n)
            ] += gamma


            # Im(C_n)

            if n >= 1:

                A[
                    row,
                    idx_ImC(n)
                ] += (
                    -2.0
                    * g
                    * np.sqrt(n)
                )


            # G_{n+1}

            if n < Nmax:

                A[
                    row,
                    idx_G(n + 1)
                ] += (
                    kappa
                    * (n + 1)
                )


        # ====================================================
        # ECUACIÓN (10)
        #
        # dX_n/dt
        # ====================================================

        for n in range(Nmax + 1):

            row = idx_X(n)


            # G_n

            A[
                row,
                idx_G(n)
            ] += P


            # X_n

            A[
                row,
                idx_X(n)
            ] += (
                -gamma
                - kappa * n
            )


            # Im(C_{n+1})

            if n < Nmax:

                A[
                    row,
                    idx_ImC(n + 1)
                ] += (
                    2.0
                    * g
                    * np.sqrt(n + 1)
                )


            # X_{n+1}

            if n < Nmax:

                A[
                    row,
                    idx_X(n + 1)
                ] += (
                    kappa
                    * (n + 1)
                )


        # ====================================================
        # ECUACIÓN (11)
        # ====================================================

        for n in range(1, Nmax + 1):

            Gamma_n = (

                gamma

                + kappa
                * (2 * n - 1)

                + P

            ) / 2.0


            # =================================================
            # PARTE REAL DE C_n
            # =================================================

            row_Re = idx_ReC(n)


            # ReC_n

            A[
                row_Re,
                idx_ReC(n)
            ] += -Gamma_n


            # ImC_n

            A[
                row_Re,
                idx_ImC(n)
            ] += -Delta


            # ReC_{n+1}

            if n < Nmax:

                A[
                    row_Re,
                    idx_ReC(n + 1)
                ] += (

                    kappa

                    * np.sqrt(
                        n * (n + 1)
                    )
                )


            # =================================================
            # PARTE IMAGINARIA DE C_n
            # =================================================

            row_Im = idx_ImC(n)


            # G_n

            A[
                row_Im,
                idx_G(n)
            ] += (
                g
                * np.sqrt(n)
            )


            # X_{n-1}

            A[
                row_Im,
                idx_X(n - 1)
            ] += (
                -g
                * np.sqrt(n)
            )


            # ReC_n

            A[
                row_Im,
                idx_ReC(n)
            ] += Delta


            # ImC_n

            A[
                row_Im,
                idx_ImC(n)
            ] += -Gamma_n


            # ImC_{n+1}

            if n < Nmax:

                A[
                    row_Im,
                    idx_ImC(n + 1)
                ] += (

                    kappa

                    * np.sqrt(
                        n * (n + 1)
                    )
                )


        # ====================================================
        # 6. NORMALIZACIÓN
        #
        # sum_n (G_n + X_n) = 1
        # ====================================================

        b = np.zeros(dim)

        normalization_row = 0


        # Borrar una ecuación

        A[
            normalization_row,
            :
        ] = 0.0


        # Reemplazar por Tr(rho)=1

        for n in range(Nmax + 1):

            A[
                normalization_row,
                idx_G(n)
            ] = 1.0

            A[
                normalization_row,
                idx_X(n)
            ] = 1.0


        b[
            normalization_row
        ] = 1.0


        # ====================================================
        # 7. RESOLVER SISTEMA ESTACIONARIO
        # ====================================================

        A = A.tocsr()


        y_ss = spsolve(
            A,
            b
        )


        if not np.all(
            np.isfinite(y_ss)
        ):

            return (
                np.nan,
                np.nan,
                np.nan
            )


        # ====================================================
        # 8. EXTRAER POBLACIONES
        # ====================================================

        G = np.array([

            y_ss[idx_G(n)]

            for n in range(
                Nmax + 1
            )

        ])


        X = np.array([

            y_ss[idx_X(n)]

            for n in range(
                Nmax + 1
            )

        ])


        # ====================================================
        # 9. DISTRIBUCIÓN FOTÓNICA
        # ====================================================

        p_n = G + X


        n_values = np.arange(
            Nmax + 1
        )


        # ====================================================
        # 10. NÚMERO MEDIO DE FOTONES
        #
        # <n> = sum_n n p_n
        # ====================================================

        Nph = np.sum(

            n_values
            * p_n
        )


        # ====================================================
        # 11. NUMERADOR DE g2
        #
        # <n(n-1)>
        # ====================================================

        numerator = np.sum(

            n_values
            * (n_values - 1)
            * p_n
        )


        # ====================================================
        # 12. g^(2)(0)
        #
        # g2 =
        #
        # <n(n-1)>
        # -----------
        #    <n>^2
        # ====================================================

        if Nph > 1e-12:

            g2 = (
                numerator
                / Nph**2
            )

        else:

            # Si no hay prácticamente fotones,
            # g2 queda numéricamente indefinido.

            g2 = np.nan


        # ====================================================
        # 13. POBLACIÓN EN EL BORDE
        # ====================================================

        p_border = (

            G[Nmax]
            + X[Nmax]
        )


        return (
            g2,
            Nph,
            p_border
        )


    # ========================================================
    # 14. RECORRER LA MALLA
    # ========================================================

    for iP, P_meV in enumerate(
        P_values_meV
    ):


        if mostrar_progreso:

            print(
                f"Fila {iP + 1}/{n_P}"
                f" - P = {P_meV:.4g} meV"
            )


        for ik, kappa_meV in enumerate(
            kappa_values_meV
        ):


            (
                g2,
                Nph,
                p_border

            ) = resolver_estado_estacionario(

                kappa_meV,
                P_meV
            )


            g2_map[
                iP,
                ik
            ] = g2


            Nph_map[
                iP,
                ik
            ] = Nph


            border_population[
                iP,
                ik
            ] = p_border


    # ========================================================
    # 15. MALLA PARA LA GRÁFICA
    # ========================================================

    KAPPA, PP = np.meshgrid(

        kappa_values_meV,
        P_values_meV

    )


    # ========================================================
    # 16. GRÁFICA
    # ========================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )


    color_map = ax.pcolormesh(

        KAPPA,
        PP,
        g2_map,

        shading="auto",

        cmap=cmap,

        vmin=vmin,
        vmax=vmax
    )


    # --------------------------------------------------------
    # Barra de colores
    # --------------------------------------------------------

    cbar = fig.colorbar(

        color_map,
        ax=ax

    )


    cbar.set_label(
        r"$g^{(2)}(0)$"
    )


    # ========================================================
    # 17. CONTORNOS
    # ========================================================

    if contours is not None:

        contour_lines = ax.contour(

            KAPPA,
            PP,
            g2_map,

            levels=contours,
            colors="black",

            linewidths=1.0
        )


        ax.clabel(

            contour_lines,

            inline=True,

            fontsize=8
        )


    # ========================================================
    # 18. CONTORNO ESPECIAL g2 = 1
    #
    # Poissoniano
    # ========================================================

    poisson_line = ax.contour(

        KAPPA,
        PP,
        g2_map,

        levels=[1.0],
        colors="red",

        linewidths=2.0,

        linestyles="--"
    )


    ax.clabel(

        poisson_line,

        fmt={
            1.0: r"$g^{(2)}=1$"
        },

        fontsize=9
    )


    # ========================================================
    # 19. EJES
    # ========================================================

    ax.set_xlim(
        xlim
    )

    ax.set_ylim(
        ylim
    )


    ax.set_xlabel(
        r"$\kappa$ (meV)"
    )

    ax.set_ylabel(
        r"$P$ (meV)"
    )


    # ========================================================
    # 20. TÍTULO
    # ========================================================

    ax.set_title(

        r"Función de coherencia de segundo orden"
        "\n"

        rf"$g={g_meV}$ meV, "
        rf"$\Delta={Delta_meV}$ meV, "
        rf"$\gamma={gamma_meV}$ meV"
    )


    fig.tight_layout()

    plt.show()


    # ========================================================
    # 21. COMPROBAR TRUNCAMIENTO
    # ========================================================

    max_border = np.nanmax(
        np.abs(border_population)
    )


    print(
        "\nMáxima población encontrada "
        f"en n = {Nmax}: "
        f"{max_border:.3e}"
    )


    if max_border > 1e-5:

        print(
            "ADVERTENCIA: la población en el borde "
            "no es despreciable.\n"
            "Conviene aumentar Nmax."
        )


    # ========================================================
    # 22. INFORMACIÓN ADICIONAL
    # ========================================================

    finite_g2 = g2_map[
        np.isfinite(g2_map)
    ]


    if finite_g2.size > 0:

        print(
            "g2 mínimo =",
            np.min(finite_g2)
        )

        print(
            "g2 máximo =",
            np.max(finite_g2)
        )


    # ========================================================
    # 23. DEVOLVER RESULTADOS
    # ========================================================

    datos = {

        "kappa":
            kappa_values_meV,

        "P":
            P_values_meV,

        "g2":
            g2_map,

        "Nph":
            Nph_map,

        "border_population":
            border_population,

        "KAPPA_mesh":
            KAPPA,

        "P_mesh":
            PP
    }


    return (
        fig,
        ax,
        datos
    )

In [ ]:
fig, ax, datos = mapa_g2(

    # Parámetros físicos
    Delta_meV=0,
    g_meV=1,
    gamma_meV=0.1,

    # Eje x: kappa
    xlim=(0.1, 2.6),

    # Eje y: P
    ylim=(0.1, 50),

    # Truncamiento
    Nmax=100,

    # Resolución
    n_kappa=100,
    n_P=100,

    # Escala de colores
    vmin=0,
    vmax=2,

    # Curvas de nivel
    contours=10
)

In [ ]:
fig, ax, datos = mapa_g2(

    # Parámetros físicos
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,

    # Eje x: kappa
    xlim=(0.1, 2.6),

    # Eje y: P
    ylim=(0.1, 50),

    # Truncamiento
    Nmax=100,

    # Resolución
    n_kappa=100,
    n_P=100,

    # Escala de colores
    vmin=0,
    vmax=2,

    # Curvas de nivel
    contours=10
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp


# Se asume que ya tienes definido:
#
# HBAR = 0.6582119569  # meV*ps
#
# y que ya compilaste:
#
# resolver_y_graficar(...)


def resolver_correlacion_QRT(
    datos_rho,
    omegaX_meV,
    Delta_meV,
    g_meV,
    gamma_meV,
    kappa_meV,
    P_meV,
    canal="cavidad",
    tau_final=10.0,
    num_points=5000,
    Nmax=None,
    graficar=True,
    xlim=None,
    ylim=None,
    figsize=(10, 6)
):
    """
    Resuelve las ecuaciones corregidas de evolución necesarias
    para aplicar el teorema de regresión cuántica.

    ----------------------------------------------------------
    CANALES
    ----------------------------------------------------------

    canal="cavidad"

        O(t) = a(t)

        Calcula:

        G_C^(1)(tau)
        = <a†(t+tau) a(t)>


    canal="exciton"

        O(t) = sigma(t)

        Calcula:

        G_X^(1)(tau)
        = <sigma†(t+tau) sigma(t)>


    ----------------------------------------------------------
    DATOS DE ENTRADA
    ----------------------------------------------------------

    datos_rho:

        Diccionario producido por:

        fig, ax, datos_rho = resolver_y_graficar(...)

        Se utiliza el último instante calculado como
        aproximación al estado estacionario.


    omegaX_meV:
        Energía del excitón [meV]

    Delta_meV:
        Detuning [meV]

    g_meV:
        Acoplamiento excitón-fotón [meV]

    gamma_meV:
        Pérdida excitónica hacia modos leaky [meV]

    kappa_meV:
        Pérdida de cavidad [meV]

    P_meV:
        Bombeo incoherente [meV]

    tau_final:
        Máximo retardo tau [ps]

    ----------------------------------------------------------
    CONVENCIÓN DE ARRAYS
    ----------------------------------------------------------

    AG[n] = <a_Gn†(t+tau) O(t)>

    AX[n] = <a_Xn†(t+tau) O(t)>

    SIG[n] = <sigma_n†(t+tau) O(t)>

    VAR[n] = <varsigma_n(t+tau) O(t)>

    Esta convención evita que AX[n] represente a_X,n-1†.

    ----------------------------------------------------------
    DEVUELVE
    ----------------------------------------------------------

    resultados["tau"]
    resultados["G1"]
    resultados["G1_rot"]

    resultados["AG_rot"]
    resultados["AX_rot"]
    resultados["SIG_rot"]
    resultados["VAR_rot"]

    etc.
    """


    # ========================================================
    # 1. TOMAR ESTADO ESTACIONARIO DE rho
    # ========================================================

    G_all = datos_rho["G"]
    X_all = datos_rho["X"]

    ReC_all = datos_rho["ReC"]
    ImC_all = datos_rho["ImC"]


    # Último instante calculado

    G = np.array(
        G_all[:, -1],
        dtype=float
    )

    X = np.array(
        X_all[:, -1],
        dtype=float
    )

    ReC = np.array(
        ReC_all[:, -1],
        dtype=float
    )

    ImC = np.array(
        ImC_all[:, -1],
        dtype=float
    )


    # ========================================================
    # 2. DETERMINAR Nmax
    # ========================================================

    Nmax_datos = len(G) - 1


    if Nmax is None:

        Nmax = Nmax_datos


    if Nmax > Nmax_datos:

        raise ValueError(
            "Nmax es mayor que el usado para calcular "
            "datos_rho."
        )


    # Recortar si el usuario pide Nmax menor

    G = G[:Nmax + 1]
    X = X[:Nmax + 1]


    # ========================================================
    # 3. CONSTRUIR COHERENCIAS
    #
    # C[n] = rho_{Gn,X(n-1)}
    # ========================================================

    C = np.zeros(
        Nmax + 1,
        dtype=complex
    )


    for n in range(1, Nmax + 1):

        C[n] = (

            ReC[n - 1]

            + 1j
            * ImC[n - 1]
        )


    # ========================================================
    # 4. CONVERSIÓN DE UNIDADES
    # ========================================================

    omegaX = omegaX_meV / HBAR

    Delta = Delta_meV / HBAR

    g = g_meV / HBAR

    gamma = gamma_meV / HBAR

    kappa = kappa_meV / HBAR

    P = P_meV / HBAR


    # Frecuencia del modo desnudo de cavidad

    omegaC = (
        omegaX
        - Delta
    )


    # ========================================================
    # 5. DIMENSIONES DE LOS ARRAYS
    # ========================================================

    # AG[n] y AX[n]:
    #
    # a_Gn† = |G,n+1><G,n|
    #
    # requieren n+1 <= Nmax
    #
    # por tanto:
    #
    # n = 0,...,Nmax-1

    Nphot_transition = Nmax


    # SIG[n]:
    #
    # sigma_n† = |X,n><G,n|
    #
    # n = 0,...,Nmax

    Nsigma = Nmax + 1


    # VAR[n] se guardará con tamaño Nmax+1
    # aunque físicamente solo n=1,...,Nmax-1
    # son necesarios.

    Nvar = Nmax + 1


    # ========================================================
    # 6. CONDICIONES INICIALES
    # ========================================================

    AG0 = np.zeros(
        Nphot_transition,
        dtype=complex
    )

    AX0 = np.zeros(
        Nphot_transition,
        dtype=complex
    )

    SIG0 = np.zeros(
        Nsigma,
        dtype=complex
    )

    VAR0 = np.zeros(
        Nvar,
        dtype=complex
    )


    canal = canal.lower()


    # ========================================================
    # CANAL DE CAVIDAD
    #
    # Condiciones iniciales de la Ec. (26)
    # ========================================================

    if canal in [
        "cavidad",
        "cavity"
    ]:


        # ----------------------------------------------------
        # <a_Gn†(t) a(t)>
        #
        # = sqrt(n+1) rho_G(n+1),G(n+1)
        #
        # n = 0,...,Nmax-1
        # ----------------------------------------------------

        for n in range(
            0,
            Nmax
        ):

            AG0[n] = (

                np.sqrt(n + 1)

                * G[n + 1]
            )


        # ----------------------------------------------------
        # <a_Xn†(t) a(t)>
        #
        # Para la Ec. (26):
        #
        # <a_X,n-1† a>
        # = sqrt(n) rho_Xn,Xn
        #
        # Sustituyendo m=n-1:
        #
        # <a_Xm† a>
        # = sqrt(m+1) rho_X,m+1;X,m+1
        #
        # Por eso:
        #
        # AX0[m] = sqrt(m+1) X[m+1]
        # ----------------------------------------------------

        for n in range(
            0,
            Nmax
        ):

            AX0[n] = (

                np.sqrt(n + 1)

                * X[n + 1]
            )


        # ----------------------------------------------------
        # <sigma_n†(t) a(t)>
        #
        # = sqrt(n+1)
        #   rho_G(n+1),Xn
        #
        # = sqrt(n+1) C[n+1]
        #
        # Tomamos t=0 como origen del estado estacionario,
        # por lo que exp(-i Delta t)=1.
        # ----------------------------------------------------

        for n in range(
            0,
            Nmax
        ):

            SIG0[n] = (

                np.sqrt(n + 1)

                * C[n + 1]
            )


        # ----------------------------------------------------
        # <varsigma_n(t) a(t)>
        #
        # = sqrt(n)
        #   rho_Xn,G(n+1)
        #
        # rho_Xn,G(n+1)
        # = C[n+1]^*
        #
        # n >= 1
        # ----------------------------------------------------

        for n in range(
            1,
            Nmax
        ):

            VAR0[n] = (

                np.sqrt(n)

                * np.conj(
                    C[n + 1]
                )
            )


    # ========================================================
    # CANAL EXCITÓNICO
    #
    # Condiciones iniciales de la Ec. (27)
    # ========================================================

    elif canal in [
        "exciton",
        "x"
    ]:


        # ----------------------------------------------------
        # <a_Gn† sigma>
        #
        # = rho_Xn,G(n+1)
        #
        # = C[n+1]^*
        # ----------------------------------------------------

        for n in range(
            0,
            Nmax
        ):

            AG0[n] = np.conj(
                C[n + 1]
            )


        # ----------------------------------------------------
        # <a_Xn† sigma> = 0
        # ----------------------------------------------------

        AX0[:] = 0.0


        # ----------------------------------------------------
        # <sigma_n† sigma>
        #
        # = rho_Xn,Xn
        # ----------------------------------------------------

        SIG0[:] = X


        # ----------------------------------------------------
        # <varsigma_n sigma> = 0
        # ----------------------------------------------------

        VAR0[:] = 0.0


    else:

        raise ValueError(
            "canal debe ser 'cavidad' o 'exciton'."
        )


    # ========================================================
    # 7. EMPAQUETAR VECTOR
    # ========================================================

    nAG = len(AG0)
    nAX = len(AX0)
    nSIG = len(SIG0)
    nVAR = len(VAR0)


    y0 = np.concatenate(
        [
            AG0,
            AX0,
            SIG0,
            VAR0
        ]
    )


    # ========================================================
    # 8. FUNCIÓN PARA DESEMPAQUETAR
    # ========================================================

    def unpack(y):

        p0 = 0

        p1 = p0 + nAG

        AG = y[
            p0:p1
        ]


        p0 = p1

        p1 = p0 + nAX

        AX = y[
            p0:p1
        ]


        p0 = p1

        p1 = p0 + nSIG

        SIG = y[
            p0:p1
        ]


        p0 = p1

        p1 = p0 + nVAR

        VAR = y[
            p0:p1
        ]


        return (
            AG,
            AX,
            SIG,
            VAR
        )


    # ========================================================
    # 9. SISTEMA CORREGIDO DE ECUACIONES
    # ========================================================
    #
    # Trabajamos en un marco que rota con:
    #
    # omegaC = omegaX - Delta
    #
    # Por eso:
    #
    # a_Gn† -> frecuencia diagonal 0
    #
    # a_Xn† -> frecuencia diagonal 0
    #
    # sigma_n† -> +Delta
    #
    # varsigma_n -> -Delta
    #
    # ya que:
    #
    # (omegaX - 2Delta)
    # - (omegaX - Delta)
    # = -Delta
    # ========================================================

    def sistema_QRT(
        tau,
        y
    ):


        (
            AG,
            AX,
            SIG,
            VAR

        ) = unpack(y)


        dAG = np.zeros_like(AG)

        dAX = np.zeros_like(AX)

        dSIG = np.zeros_like(SIG)

        dVAR = np.zeros_like(VAR)


        # ====================================================
        # ECUACIÓN 1
        #
        # d/dtau <a_Gn†>
        #
        # CORRECCIÓN:
        #
        # + gamma <a_Xn†>
        #
        # NO:
        #
        # + gamma <a_X,n-1†>
        # ====================================================

        for n in range(
            0,
            Nmax
        ):


            dAG[n] = (

                # --------------------------------------------
                # [ -kappa/2 (2n+1) - P ] AG_n
                # --------------------------------------------

                AG[n]
                * (
                    - kappa
                    * (2*n + 1)
                    / 2

                    - P
                )


                # --------------------------------------------
                # + i g sqrt(n+1) sigma_n
                # --------------------------------------------

                + 1j
                * g
                * np.sqrt(n + 1)
                * SIG[n]


                # --------------------------------------------
                # + gamma a_Xn†
                # CORREGIDO
                # --------------------------------------------

                + gamma
                * AX[n]
            )


            # -----------------------------------------------
            # + kappa sqrt((n+1)(n+2))
            #   a_G,n+1†
            # -----------------------------------------------

            if n < Nmax - 1:

                dAG[n] += (

                    kappa

                    * np.sqrt(
                        (n + 1)
                        * (n + 2)
                    )

                    * AG[n + 1]
                )


            # -----------------------------------------------
            # - i g sqrt(n) varsigma_n
            # -----------------------------------------------

            if n >= 1:

                dAG[n] += (

                    - 1j
                    * g
                    * np.sqrt(n)
                    * VAR[n]
                )


        # ====================================================
        # ECUACIÓN 2
        #
        # d/dtau <sigma_n†>
        # ====================================================

        for n in range(
            0,
            Nmax + 1
        ):


            # -----------------------------------------------
            # Diagonal
            #
            # En laboratorio:
            #
            # i omegaX
            #
            # En marco omegaC:
            #
            # i Delta
            # -----------------------------------------------

            dSIG[n] = (

                SIG[n]
                * (
                    1j * Delta

                    - (
                        gamma
                        + P
                    ) / 2

                    - kappa * n
                )
            )


            # -----------------------------------------------
            # + i g sqrt(n+1) a_Gn†
            # -----------------------------------------------

            if n < Nmax:

                dSIG[n] += (

                    1j
                    * g
                    * np.sqrt(n + 1)
                    * AG[n]
                )


            # -----------------------------------------------
            # + kappa(n+1) sigma_n+1†
            # -----------------------------------------------

            if n < Nmax:

                dSIG[n] += (

                    kappa
                    * (n + 1)
                    * SIG[n + 1]
                )


            # -----------------------------------------------
            # - i g sqrt(n) a_X,n-1†
            #
            # a_X,n-1† = AX[n-1]
            # -----------------------------------------------

            if n >= 1:

                dSIG[n] += (

                    -1j
                    * g
                    * np.sqrt(n)
                    * AX[n - 1]
                )


        # ====================================================
        # ECUACIÓN 3
        #
        # Originalmente escrita para:
        #
        # a_X,n-1†
        #
        # En nuestro código usamos:
        #
        # m = n-1
        #
        # y AX[m] = a_Xm†
        #
        #
        # CORRECCIÓN IMPORTANTE:
        #
        # P a_G,n-1†
        #
        # Por tanto, usando m=n-1:
        #
        # P a_Gm†
        #
        # -> P * AG[m]
        # ====================================================

        for m in range(
            0,
            Nmax
        ):


            # n del paper es:
            #
            # n = m + 1

            n = m + 1


            dAX[m] = (

                # --------------------------------------------
                # + P a_G,n-1†
                #
                # = P a_Gm†
                #
                # CORREGIDO
                # --------------------------------------------

                P
                * AG[m]


                # --------------------------------------------
                # - i g sqrt(n) sigma_n†
                # --------------------------------------------

                - 1j
                * g
                * np.sqrt(n)
                * SIG[n]


                # --------------------------------------------
                # diagonal
                #
                # i(omegaX-Delta) desaparece
                # en el marco omegaC.
                # --------------------------------------------

                + AX[m]
                * (
                    - gamma

                    - kappa
                    * (2*n - 1)
                    / 2
                )
            )


            # -----------------------------------------------
            # + kappa sqrt(n(n+1))
            #   a_Xn†
            #
            # a_Xn† = AX[m+1]
            # -----------------------------------------------

            if m < Nmax - 1:

                dAX[m] += (

                    kappa
                    * np.sqrt(
                        n
                        * (n + 1)
                    )

                    * AX[m + 1]
                )


            # -----------------------------------------------
            # + i g sqrt(n+1) varsigma_n
            # -----------------------------------------------

            if n < Nmax:

                dAX[m] += (

                    1j
                    * g
                    * np.sqrt(n + 1)
                    * VAR[n]
                )


        # ====================================================
        # ECUACIÓN 4
        #
        # d/dtau <varsigma_n>
        #
        # CORRECCIÓN:
        #
        # frecuencia:
        #
        # omegaX - 2 Delta
        #
        # En marco omegaC = omegaX-Delta:
        #
        # queda:
        #
        # -Delta
        # ====================================================

        for n in range(
            1,
            Nmax
        ):


            dVAR[n] = (

                # --------------------------------------------
                # - i g sqrt(n) a_Gn†
                # --------------------------------------------

                -1j
                * g
                * np.sqrt(n)
                * AG[n]


                # --------------------------------------------
                # + i g sqrt(n+1) a_X,n-1†
                #
                # = AX[n-1]
                # --------------------------------------------

                + 1j
                * g
                * np.sqrt(n + 1)
                * AX[n - 1]


                # --------------------------------------------
                # diagonal
                #
                # i(omegaX-2Delta)
                #
                # -> -i Delta en marco omegaC
                # --------------------------------------------

                + VAR[n]
                * (
                    -1j * Delta

                    - (
                        gamma
                        + P
                    ) / 2

                    - kappa * n
                )
            )


            # -----------------------------------------------
            # + kappa sqrt(n(n+2))
            #   varsigma_n+1
            # -----------------------------------------------

            if n < Nmax - 1:

                dVAR[n] += (

                    kappa

                    * np.sqrt(
                        n
                        * (n + 2)
                    )

                    * VAR[n + 1]
                )


        return np.concatenate(
            [
                dAG,
                dAX,
                dSIG,
                dVAR
            ]
        )


    # ========================================================
    # 10. VECTOR DE TAU
    # ========================================================

    tau = np.linspace(
        0.0,
        tau_final,
        num_points
    )


    # ========================================================
    # 11. RESOLVER EDO
    # ========================================================

    sol = solve_ivp(

        sistema_QRT,

        t_span=(
            0.0,
            tau_final
        ),

        y0=y0,

        t_eval=tau,

        method="BDF",

        rtol=1e-8,

        atol=1e-10
    )


    if not sol.success:

        raise RuntimeError(
            "Falló la integración QRT: "
            + sol.message
        )


    # ========================================================
    # 12. EXTRAER SOLUCIÓN
    # ========================================================

    (
        AG_rot,
        AX_rot,
        SIG_rot,
        VAR_rot

    ) = unpack(
        sol.y
    )


    # ========================================================
    # 13. CONSTRUIR G_C^(1) O G_X^(1)
    # ========================================================


    # --------------------------------------------------------
    # CAVIDAD
    #
    # Ec. (18)
    #
    # G_C^(1)
    #
    # = sum_n sqrt(n+1)
    #
    #   [AG[n] + AX[n]]
    #
    # Esta es una ventaja de nuestra nueva convención:
    #
    # AG[n] y AX[n] llevan exactamente el mismo índice.
    # --------------------------------------------------------

    if canal in [
        "cavidad",
        "cavity"
    ]:


        G1_rot = np.zeros(
            len(tau),
            dtype=complex
        )


        for n in range(
            0,
            Nmax
        ):

            G1_rot += (

                np.sqrt(n + 1)

                * (
                    AG_rot[n]
                    + AX_rot[n]
                )
            )


    # --------------------------------------------------------
    # EXCITÓN
    #
    # Ec. (19)
    #
    # G_X^(1)
    #
    # = sum_n SIG[n]
    # --------------------------------------------------------

    else:

        G1_rot = np.sum(
            SIG_rot,
            axis=0
        )


    # ========================================================
    # 14. RESTAURAR FRECUENCIA ÓPTICA
    # ========================================================

    # Todo el sistema fue integrado después de quitar
    # la frecuencia común:
    #
    # omegaC = omegaX - Delta
    #
    # Por tanto restauramos:
    #
    # exp(+i omegaC tau)

    optical_phase = np.exp(

        1j
        * omegaC
        * tau
    )


    G1 = (

        G1_rot
        * optical_phase
    )


    # ========================================================
    # 15. CHEQUEOS EN tau = 0
    # ========================================================

    n_values = np.arange(
        Nmax + 1
    )


    Nph = np.sum(

        n_values

        * (
            G + X
        )
    )


    NX = np.sum(
        X
    )


    print(
        "\n========================================"
    )

    print(
        "CHEQUEO DE CONDICIONES INICIALES"
    )

    print(
        "========================================"
    )


    print(
        "G^(1)(0) obtenido =",
        G1[0]
    )


    if canal in [
        "cavidad",
        "cavity"
    ]:

        print(
            "<a†a> = Nph =",
            Nph
        )

        print(
            "Diferencia =",
            abs(
                G1[0] - Nph
            )
        )


    else:

        print(
            "<sigma†sigma> =",
            NX
        )

        print(
            "Diferencia =",
            abs(
                G1[0] - NX
            )
        )


    # ========================================================
    # 16. GRÁFICA
    # ========================================================

    if graficar:


        fig, ax = plt.subplots(
            figsize=figsize
        )


        ax.plot(

            tau,

            np.real(G1),

            label="Parte real"
        )


        ax.plot(

            tau,

            np.imag(G1),

            "--",

            label="Parte imaginaria"
        )


        if xlim is None:

            ax.set_xlim(
                0,
                tau_final
            )

        else:

            ax.set_xlim(
                xlim
            )


        if ylim is not None:

            ax.set_ylim(
                ylim
            )


        ax.set_xlabel(
            r"$\tau$ (ps)"
        )


        if canal in [
            "cavidad",
            "cavity"
        ]:

            ax.set_ylabel(
                r"$G_C^{(1)}(\tau)$"
            )

            ax.set_title(
                "Correlación de primer orden de la cavidad"
            )


        else:

            ax.set_ylabel(
                r"$G_X^{(1)}(\tau)$"
            )

            ax.set_title(
                "Correlación de primer orden excitónica"
            )


        ax.legend()

        ax.grid(
            alpha=0.3
        )

        fig.tight_layout()

        plt.show()


    # ========================================================
    # 17. DEVOLVER RESULTADOS
    # ========================================================

    resultados = {

        "tau":
            tau,

        # Correlación física
        "G1":
            G1,

        # Correlación en marco rotante
        "G1_rot":
            G1_rot,

        "AG_rot":
            AG_rot,

        "AX_rot":
            AX_rot,

        "SIG_rot":
            SIG_rot,

        "VAR_rot":
            VAR_rot,

        "G_ss":
            G,

        "X_ss":
            X,

        "C_ss":
            C,

        "Nph":
            Nph,

        "NX":
            NX,

        "omegaC_meV":
            omegaX_meV - Delta_meV,

        "solution":
            sol
    }


    return resultados

In [ ]:
fig, ax, datos_rho = resolver_y_graficar(

    g_meV=1,
    Delta_meV=5,
    gamma_meV=0.1,
    kappa_meV=5,
    P_meV=1,

    t_initial=0,
    t_final=200,

    n_to_plot=[0, 5, 10, 15, 20, 25, 30, 35],

    xlim=(0, 200),
    ylim=(0, 0.1),

    Nmax=100,

    num_points=2001,

    mostrar_leyenda=False
)

In [ ]:
resultado_C = resolver_correlacion_QRT(

    datos_rho=datos_rho,

    omegaX_meV=1000,

    Delta_meV=5,

    g_meV=1,

    gamma_meV=0.1,

    kappa_meV=0.1,

    P_meV=15,

    canal="cavidad",

    tau_final=100,

    num_points=5000,

    Nmax=100,

    xlim=(0, 10)
)


In [ ]:
resultado_X = resolver_correlacion_QRT(

    datos_rho=datos_rho,

    omegaX_meV=1000,

    Delta_meV=5,

    g_meV=1,

    gamma_meV=0.1,

    kappa_meV=0.1,

    P_meV=15,

    canal="exciton",

    tau_final=10,

    num_points=5000,

    Nmax=100,

    xlim=(0, 10)
)

In [ ]:
"""Espectro estacionario obtenido a partir de correlaciones temporales explícitas.

Requiere NumPy, SciPy y Matplotlib. Pegar completo en una celda de Colab.
1. Calcula rho_ss con los parámetros de la llamada.
2. Integra dY/dtau = M Y mediante BDF, para los dos canales.
3. Reconstruye G_C^(1)(tau) y G_X^(1)(tau), en marco rotante.
4. Integra esas muestras con Simpson y una transformada de Fourier ZoomFFT.

No usa la resolvente, no ajusta lorentzianas y no elimina picos ni negativos.

Ejemplo:
    fig, ax, espectro = graficar_espectro_emision(
        omegaX_meV=1000, Delta_meV=5, g_meV=1,
        gamma_meV=0.1, kappa_meV=0.1, P_meV=15,
        Nmax=100, tau_final=5000, num_points=None,
        num_energias=10001, ventana=None,
        xlim=(990, 1000), ylim=(0, 0.01))
    datos_rho = espectro["datos_rho"]
    tau = espectro["tau_ps"]
    GC_rot = espectro["G1_cavidad_rot"]
    GX_rot = espectro["G1_exciton_rot"]
"""


def graficar_espectro_emision(
    omegaX_meV=1000.0,
    Delta_meV=5.0,
    g_meV=1.0,
    gamma_meV=0.1,
    kappa_meV=0.1,
    P_meV=15.0,
    Nmax=100,
    xlim=None,
    ylim=None,
    num_energias=10001,
    tau_final=5000.0,
    num_points=None,
    ventana=None,
    rtol=1e-8,
    atol=1e-11,
    tolerancia_cola=1e-4,
    exigir_decaimiento=True,
    ponderar_emision=True,
    normalizar=True,
    figsize=(10, 6),
    mostrar=True,
    informar=True,
):
    """Calcula rho_ss, las dos correlaciones temporales y sus espectros.

    Parámetros físicos en meV: omegaX_meV, Delta_meV, g_meV, gamma_meV,
    kappa_meV y P_meV. E_cavidad = omegaX_meV - Delta_meV.
    Nmax es el máximo número de fotones; se comprueba la población del borde.

    tau_final (ps):
        Tiempo máximo de correlación. No es el tiempo para alcanzar rho_ss.
        5000 ps permite decaer la correlación lenta del caso Fig. 10 estudiado.
        Otras condiciones pueden exigir más tiempo.
    num_points:
        Muestras temporales uniformes, contando tau=0 y tau=tau_final.
        None elige el paso con las escalas físicas y el rango de energía.
        Se ajusta hacia arriba a 4*k+1 para Simpson y su control de muestreo.
        Es la malla de salida: BDF elige sus propios pasos internos adaptativos.
    num_energias:
        Muestras de la malla espectral base, con refinamiento de los máximos.
        Más puntos interpolan mejor la transformada; no sustituyen mayor tau.
    ventana:
        None (recomendado), "hann" o "parzen". Estas ventanas de retardos
        positivos empiezan en 1 y terminan en 0. Aplicarlas cambia el espectro
        y puede ensancharlo. Nunca se usa np.hanning(num_points) sobre tau>=0.
    rtol, atol:
        Tolerancias de la integración BDF de las ecuaciones de correlación.
    tolerancia_cola:
        Umbral de max(|G|)/G(0) sobre el último 5% del intervalo temporal.
        Se comprueba en las correlaciones originales, ANTES de la ventana.
    exigir_decaimiento:
        True detiene el cálculo si la cola supera tolerancia_cola.
        False permite estudiar intervalos cortos, emitiendo una advertencia.
        La prueba de la cola es un diagnóstico, no una cota rigurosa del error.

    xlim, ylim recortan la gráfica, sin normalizar solo el fragmento visible.
    ponderar_emision aplica kappa/hbar y gamma/hbar a los canales.
    normalizar divide los dos espectros por UN MISMO máximo.
    Las intensidades sin normalizar se expresan en unidades arbitrarias.

    Devuelve fig, ax, espectro. Entre sus claves:
      energia_meV, S_cavidad, S_exciton, S_leaky, y sus versiones _raw;
      datos_rho (G, X, ReC, ImC con UNA columna estacionaria, más rho_ss);
      tau_ps, G1_cavidad_rot, G1_exciton_rot, g1_cavidad_rot, g1_exciton_rot;
      ventana_temporal, diagnostico, parametros, Nph, NX, g2_0.

    G1 son correlaciones sin normalizar; g1 = G1/G1(0), o NaN si G1(0)=0.
    Se devuelven en el marco rotante de la cavidad. En laboratorio:
        G1_lab(tau) = exp(1j*E_cavidad*tau/hbar) * G1_rot(tau).
    El muestreo automático está pensado para G1_rot, no para dibujar la
    portadora óptica de G1_lab. El espectro sí usa energías de laboratorio.

    La transformada se evalúa NUMÉRICAMENTE desde 0 hasta tau_final:
        S0(E) = Re integral[G1_rot(tau)*w(tau)
                           *exp(-1j*(E-E_cavidad)*tau/hbar) dtau] / pi.
    ZoomFFT calcula eficientemente la suma de Simpson sobre esas muestras.
    No se utiliza una resolvente ni se extrapola la cola hasta infinito.
    No se calcula abs(FFT)**2 ni se recortan intensidades negativas.
    No hacen falta otros archivos, funciones globales ni datos_rho externos.
    """
    import warnings
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.integrate import BDF
    from scipy.signal import ZoomFFT
    from scipy.optimize import minimize_scalar
    from scipy.sparse import lil_matrix, csc_matrix, block_diag
    from scipy.sparse.linalg import spsolve, MatrixRankWarning

    HBAR = 0.6582119569  # meV ps

    valores = np.asarray([omegaX_meV, Delta_meV, g_meV,
                          gamma_meV, kappa_meV, P_meV], dtype=float)
    if valores.shape != (6,) or not np.isfinite(valores).all():
        raise ValueError("Los parámetros físicos deben ser escalares finitos.")
    omegaX_meV, Delta_meV, g_meV, gamma_meV, kappa_meV, P_meV = valores
    if gamma_meV < 0 or P_meV < 0 or kappa_meV <= 0:
        raise ValueError("Se requiere gamma_meV >= 0, P_meV >= 0 y kappa_meV > 0.")
    for nombre, valor, minimo in (("Nmax", Nmax, 1),
                                   ("num_energias", num_energias, 101)):
        if (not np.isscalar(valor) or not np.isfinite(valor)
                or int(valor) != valor or valor < minimo):
            raise ValueError(f"{nombre} debe ser un entero >= {minimo}.")
    Nmax, num_energias = int(Nmax), int(num_energias)
    for nombre, limites in (("xlim", xlim), ("ylim", ylim)):
        if limites is not None:
            v = np.asarray(limites, dtype=float)
            if v.shape != (2,) or not np.isfinite(v).all() or v[0] >= v[1]:
                raise ValueError(f"{nombre} debe ser (mínimo, máximo), con mínimo < máximo.")

    for nombre, valor in (("tau_final", tau_final), ("rtol", rtol),
                           ("atol", atol), ("tolerancia_cola", tolerancia_cola)):
        if not np.isscalar(valor) or not np.isfinite(valor) or valor <= 0:
            raise ValueError(f"{nombre} debe ser un escalar positivo finito.")
    if tolerancia_cola >= 1:
        raise ValueError("tolerancia_cola debe estar entre 0 y 1.")
    if num_points is not None:
        if (not np.isscalar(num_points) or not np.isfinite(num_points)
                or int(num_points) != num_points or num_points < 5):
            raise ValueError("num_points debe ser un entero >= 5 o None.")
        num_points = int(num_points)
    if ventana not in (None, "hann", "parzen"):
        raise ValueError('ventana debe ser None, "hann" o "parzen".')
    if ventana is not None:
        warnings.warn("La ventana modifica la forma y anchura del espectro. "
                      "Para el espectro sin ensanchamiento artificial usa ventana=None.",
                      RuntimeWarning, stacklevel=2)

    def resolver_estado_estacionario(
        Delta_meV=5., g_meV=1., gamma_meV=.1, kappa_meV=.1, P_meV=15., Nmax=100,
    ):
        """Resuelve las ecuaciones 9-11 con traza uno, sin integración temporal."""
        N = int(Nmax)
        if N != Nmax or N < 1:
            raise ValueError('Nmax debe ser un entero positivo.')
        D, g, gamma, kappa, P = np.array(
            [Delta_meV, g_meV, gamma_meV, kappa_meV, P_meV], float)/HBAR
        if not np.isfinite([D, g, gamma, kappa, P]).all() or min(gamma,kappa,P) < 0:
            raise ValueError('Parámetros no finitos o tasas negativas.')
        # y = (G[0:N+1], X[0:N+1], ReC[1:N+1], ImC[1:N+1]).
        def rhs(y):
            G, X = y[:N+1], y[N+1:2*N+2]
            R, I = y[2*N+2:3*N+2], y[3*N+2:]
            n = np.arange(N+1, dtype=float)
            j = np.arange(1, N+1, dtype=float)
            if y.ndim == 2:
                n, j = n[:, None], j[:, None]
            sq = np.sqrt(j)
            dG = gamma*X-P*G-kappa*n*G
            dX = P*G-gamma*X-kappa*n*X
            dG[:-1] += kappa*j*G[1:]
            dX[:-1] += kappa*j*X[1:]
            dG[1:] -= 2*g*sq*I
            dX[:-1] += 2*g*sq*I
            decay = (gamma+P+kappa*(2*j-1))/2
            dR = -D*I-decay*R
            dI = D*R-decay*I+g*sq*(G[1:]-X[:-1])
            dR[:-1] += kappa*np.sqrt(j[:-1]*(j[:-1]+1))*R[1:]
            dI[:-1] += kappa*np.sqrt(j[:-1]*(j[:-1]+1))*I[1:]
            return np.concatenate((dG, dX, dR, dI))
        L = csc_matrix(rhs(np.eye(4*N+2))).tolil()
        trace = np.r_[np.ones(2*N+2), np.zeros(2*N)]
        L[0] = trace
        b = np.zeros(4*N+2)
        b[0] = 1.
        y = spsolve(L.tocsc(), b)
        if not np.isfinite(y).all() or np.max(np.abs(rhs(y))) > 1e-8:
            raise RuntimeError('No se obtuvo un estado estacionario único y convergido.')
        return dict(G=y[:N+1,None], X=y[N+1:2*N+2,None],
                    ReC=y[2*N+2:3*N+2,None], ImC=y[3*N+2:,None])

    def _modelo_qrt(datos_rho, Delta_meV, g_meV, gamma_meV, kappa_meV, P_meV,
                    Nmax=None):
        """Ecuaciones QRT consistentes con la ecuación maestra, en marco rotante."""
        arrays = {k: np.asarray(datos_rho[k], dtype=float)
                  for k in ('G', 'X', 'ReC', 'ImC')}
        if any(a.ndim != 2 or a.shape[1] == 0 or not np.isfinite(a).all()
               for a in arrays.values()):
            raise ValueError('G, X, ReC e ImC deben ser arrays 2D finitos con tiempos.')
        if arrays['G'].shape != arrays['X'].shape:
            raise ValueError('G y X deben tener la misma forma.')
        Ndatos = arrays['G'].shape[0] - 1
        N = Ndatos if Nmax is None else int(Nmax)
        if N < 1 or N > Ndatos or (Nmax is not None and N != Nmax):
            raise ValueError('Nmax debe ser entero, entre 1 y el Nmax de datos_rho.')
        if any(arrays[k].shape[0] < N for k in ('ReC', 'ImC')):
            raise ValueError('Faltan coherencias para el Nmax solicitado.')
        G, X = (arrays[k][:N+1, -1].copy() for k in ('G', 'X'))
        C = np.r_[0j, arrays['ReC'][:N, -1] + 1j*arrays['ImC'][:N, -1]]
        D, g, gamma, kappa, P = np.array(
            [Delta_meV, g_meV, gamma_meV, kappa_meV, P_meV], float) / HBAR
        if not np.isfinite([D, g, gamma, kappa, P]).all():
            raise ValueError('Los parámetros físicos deben ser finitos.')
        if min(gamma, kappa, P) < 0:
            raise ValueError('gamma, kappa y P deben ser no negativos.')

        # Orden original: AG[0:N], AX[0:N], SIG[0:N+1], VAR[0:N+1].
        ag = lambda n: n
        ax = lambda n: N+n
        sig = lambda n: 2*N+n
        var = lambda n: 3*N+1+n
        M = lil_matrix((4*N+2, 4*N+2), dtype=complex)
        for n in range(N):
            M[ag(n), ag(n)] = -kappa*(2*n+1)/2 - P
            M[ag(n), sig(n)] = 1j*g*np.sqrt(n+1)
            M[ag(n), ax(n)] = gamma
            if n < N-1:
                M[ag(n), ag(n+1)] = kappa*np.sqrt((n+1)*(n+2))
            if n >= 1:
                M[ag(n), var(n)] = -1j*g*np.sqrt(n)

            M[ax(n), ag(n)] = P
            M[ax(n), sig(n+1)] = -1j*g*np.sqrt(n+1)
            M[ax(n), ax(n)] = -gamma-kappa*(2*n+1)/2
            if n < N-1:
                M[ax(n), ax(n+1)] = kappa*np.sqrt((n+1)*(n+2))
                M[ax(n), var(n+1)] = 1j*g*np.sqrt(n+2)

        for n in range(N+1):
            M[sig(n), sig(n)] = 1j*D-(gamma+P)/2-kappa*n
            if n < N:
                M[sig(n), ag(n)] = 1j*g*np.sqrt(n+1)
                M[sig(n), sig(n+1)] = kappa*(n+1)
            if n >= 1:
                M[sig(n), ax(n-1)] = -1j*g*np.sqrt(n)
        for n in range(1, N):
            M[var(n), ag(n)] = -1j*g*np.sqrt(n)
            M[var(n), ax(n-1)] = 1j*g*np.sqrt(n+1)
            M[var(n), var(n)] = -1j*D-(gamma+P)/2-kappa*n
            if n < N-1:
                M[var(n), var(n+1)] = kappa*np.sqrt(n*(n+2))

        # Dos columnas: O=a y O=sigma. No hay conjugación en c@y.
        Y0 = np.zeros((4*N+2, 2), complex)
        obs = np.zeros((2, 4*N+2), complex)
        sq = np.sqrt(np.arange(1, N+1))
        Y0[:N, 0] = sq*G[1:]
        Y0[N:2*N, 0] = sq*X[1:]
        Y0[2*N:3*N, 0] = sq*C[1:]
        Y0[3*N+2:4*N+1, 0] = np.sqrt(np.arange(1, N))*C[2:].conj()
        Y0[:N, 1] = C[1:].conj()
        Y0[2*N:3*N+1, 1] = X
        obs[0, :N] = obs[0, N:2*N] = sq
        obs[1, 2*N:3*N+1] = 1

        # Intercalar por n mantiene el Jacobiano disperso de BDF bien ordenado.
        perm = []
        for n in range(N+1):
            if n < N:
                perm.extend([ag(n), ax(n)])
            perm.append(sig(n))
            if 1 <= n < N:
                perm.append(var(n))
        perm = np.asarray(perm)
        M = M.tocsr()[perm][:, perm].tocsc()
        # Residuo de la ecuación maestra en el último estado suministrado.
        n = np.arange(N+1)
        dG = gamma*X-P*G-kappa*n*G
        dX = P*G-gamma*X-kappa*n*X
        dG[:-1] += kappa*np.arange(1, N+1)*G[1:]
        dX[:-1] += kappa*np.arange(1, N+1)*X[1:]
        intercambio = 2*g*sq*C[1:].imag
        dG[1:] -= intercambio
        dX[:-1] += intercambio
        j = np.arange(1, N+1)
        dC = (1j*D-(gamma+P)/2-kappa*(2*j-1)/2)*C[1:]
        dC += 1j*g*sq*(G[1:]-X[:-1])
        dC[:-1] += kappa*np.sqrt(j[:-1]*(j[:-1]+1))*C[2:]
        residual = np.abs(dG).sum()+np.abs(dX).sum()+2*np.abs(dC).sum()
        diagnostico = dict(traza=float((G+X).sum()),
                           poblacion_borde=float(G[-1]+X[-1]),
                           residuo_estacionario_ps_inv=float(residual))
        return dict(M=M, Y0=Y0[perm], observables=obs[:, perm],
                    perm=perm, N=N, G=G, X=X, C=C,
                    Nph=float(n@(G+X)), NX=float(X.sum()), diagnostico=diagnostico)

    def _correlaciones_temporales(modelo, tau):
        """Integra ambas columnas y guarda solo las dos correlaciones."""
        M, Y0, obs = modelo["M"], modelo["Y0"], modelo["observables"]
        dimension = M.shape[0]
        salida = np.zeros((tau.size, 2), dtype=complex)
        salida[0] = np.einsum("ij,ji->i", obs, Y0)
        if not np.any(Y0):
            return salida, dict(pasos_bdf=0, nfev=0, nlu=0)

        # Se apilan los dos problemas, independientes y con el mismo M.
        A = block_diag((M, M), format="csc")
        integrador = BDF(
            fun=lambda t, y: A @ y, t0=0.0,
            y0=Y0.T.reshape(-1), t_bound=float(tau[-1]),
            jac=A, rtol=rtol, atol=atol)
        siguiente, pasos = 1, 0
        while integrador.status == "running":
            mensaje = integrador.step()
            if integrador.status == "failed":
                raise RuntimeError(f"Falló la integración de correlaciones: {mensaje}")
            pasos += 1
            hasta = int(np.searchsorted(tau, integrador.t, side="right"))
            if hasta <= siguiente:
                continue
            interpolar = integrador.dense_output()
            # No se almacena Y(tau) completa: Nmax=100 y cientos de miles
            # de tiempos requerirían varios GB. Se proyecta cada tramo.
            while siguiente < hasta:
                fin = min(hasta, siguiente + 512)
                estados = interpolar(tau[siguiente:fin]).reshape(2, dimension, -1)
                salida[siguiente:fin] = np.einsum("cn,cnt->tc", obs, estados)
                siguiente = fin
        if siguiente != tau.size or not np.isfinite(salida).all():
            raise RuntimeError("La integración no produjo todas las correlaciones.")
        return salida, dict(pasos_bdf=pasos, nfev=integrador.nfev, nlu=integrador.nlu)

    def _pesos_simpson(numero, dt):
        pesos = np.ones(numero)
        pesos[1:-1:2] = 4.0
        pesos[2:-1:2] = 2.0
        return pesos * dt / 3.0

    # 1. Estado estacionario: L rho = 0, Tr(rho) = 1.
    parametros = dict(omegaX_meV=float(omegaX_meV), Delta_meV=float(Delta_meV),
                      g_meV=float(g_meV), gamma_meV=float(gamma_meV),
                      kappa_meV=float(kappa_meV), P_meV=float(P_meV), Nmax=Nmax)
    if P_meV == 0 and (gamma_meV > 0 or g_meV != 0):
        # Sin bombeo, el estado fundamental es la solución exacta. Evita que
        # errores de redondeo se interpreten como una emisión diminuta.
        datos_rho = dict(G=np.zeros((Nmax+1, 1)), X=np.zeros((Nmax+1, 1)),
                         ReC=np.zeros((Nmax, 1)), ImC=np.zeros((Nmax, 1)))
        datos_rho["G"][0, 0] = 1.0
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("error", MatrixRankWarning)
            try:
                datos_rho = resolver_estado_estacionario(
                    Delta_meV, g_meV, gamma_meV, kappa_meV, P_meV, Nmax)
            except MatrixRankWarning as exc:
                raise ValueError("Estos parámetros no determinan un estado estacionario único.") from exc

    modelo = _modelo_qrt(datos_rho, Delta_meV, g_meV,
                         gamma_meV, kappa_meV, P_meV, Nmax)
    diagnostico = modelo["diagnostico"].copy()
    if diagnostico["poblacion_borde"] > 1e-6:
        raise ValueError(
            f"Nmax={Nmax} es insuficiente: población del borde = "
            f"{diagnostico['poblacion_borde']:.3e}. Aumenta Nmax.")

    # Matriz completa en orden |G,0> ... |G,N>, |X,0> ... |X,N>.
    G, X, C = modelo["G"], modelo["X"], modelo["C"]
    rho_ss = np.diag(np.r_[G, X]).astype(complex)
    n = np.arange(1, Nmax + 1)
    rho_ss[n, Nmax + n] = C[1:]
    rho_ss[Nmax + n, n] = C[1:].conj()
    min_eigenvalue = float(np.linalg.eigvalsh(rho_ss).min())
    diagnostico["min_autovalor_rho"] = min_eigenvalue
    if (abs(diagnostico["traza"] - 1) > 1e-8 or min_eigenvalue < -1e-9
            or diagnostico["residuo_estacionario_ps_inv"] > 1e-7):
        raise RuntimeError("No se obtuvo una densidad estacionaria válida; revisa los parámetros y Nmax.")

    n = np.arange(Nmax + 1)
    Nph, NX = modelo["Nph"], modelo["NX"]
    g2 = float((n * (n - 1)) @ (G + X) / Nph**2) if Nph > 1e-12 else float("nan")
    datos_rho.update(rho_ss=rho_ss, parametros=parametros.copy(),
                     estacionario=True, base="G0..GN,X0..XN",
                     Nph=Nph, NX=NX, g2_0=g2, traza=diagnostico["traza"],
                     p_n_final=G+X, diagnostico=diagnostico.copy())

    # 2. Correlaciones explícitas en una malla de retardos positivos.
    EC = float(omegaX_meV - Delta_meV)
    span = max(10.0, 2 * abs(g_meV) * np.sqrt(max(Nph, 0) + 1),
               gamma_meV + P_meV, kappa_meV)
    e_min = min(0.0, Delta_meV) - span
    e_max = max(0.0, Delta_meV) + span
    # Margen para los puntos que se agregarán al refinar máximos.
    paso_base = (e_max - e_min) / (num_energias - 1)
    max_energia = max(abs(e_min), abs(e_max)) + 2 * paso_base
    if xlim is not None:
        max_energia = max(max_energia, abs(xlim[0] - EC), abs(xlim[1] - EC))
    escala_temporal = max(max_energia, P_meV + gamma_meV, kappa_meV,
                         abs(Delta_meV) + 2 * abs(g_meV) * np.sqrt(max(Nph, 0) + 1))
    if num_points is None:
        dt_objetivo = min(0.02, 0.25 * HBAR / escala_temporal)
        num_points = 4 * int(np.ceil(tau_final / dt_objetivo / 4)) + 1
        if num_points > 2_000_001:
            raise ValueError(
                f"El muestreo automático requiere {num_points:,} tiempos. "
                "Revisa tau_final y los límites de energía. Si realmente necesitas "
                "esa malla y dispones de memoria, fija num_points explícitamente.")
    else:
        ajustado = 4 * int(np.ceil((num_points - 1) / 4)) + 1
        if ajustado != num_points:
            warnings.warn(f"num_points se ajustó de {num_points} a {ajustado} "
                          "para la cuadratura de Simpson.", RuntimeWarning, stacklevel=2)
        num_points = ajustado
    tau = np.linspace(0.0, float(tau_final), num_points)
    dt = float(tau[1] - tau[0])
    fase_por_paso = escala_temporal * dt / HBAR
    if fase_por_paso > 0.5:
        sugeridos = 4 * int(np.ceil(tau_final * escala_temporal / (0.25 * HBAR) / 4)) + 1
        raise ValueError(
            f"El muestreo temporal es insuficiente (dt={dt:.6g} ps). "
            f"Usa num_points=None o num_points>={sugeridos}.")
    if informar:
        print(f"Integrando las correlaciones hasta {tau_final:g} ps "
              f"({num_points:,} muestras, dt={dt:.5g} ps)...", flush=True)
    correlaciones, estadisticas_bdf = _correlaciones_temporales(modelo, tau)
    iniciales = correlaciones[0].real
    if not np.allclose(correlaciones[0], [Nph, NX], rtol=1e-8, atol=1e-10):
        raise RuntimeError("Las correlaciones en tau=0 no coinciden con Nph y NX.")
    activos = iniciales > 1e-12
    normalizadas = np.full_like(correlaciones, np.nan + 0j)
    normalizadas[:, activos] = correlaciones[:, activos] / iniciales[activos]
    inicio_cola = max(0, int(0.95 * (num_points - 1)))
    colas = np.zeros(2)
    colas[activos] = (np.max(abs(correlaciones[inicio_cola:, activos]), axis=0)
                      / iniciales[activos])
    cola_ok = bool(np.all(colas <= tolerancia_cola))
    diagnostico.update(estadisticas_bdf, dt_ps=dt, tau_final_ps=float(tau[-1]),
                      num_points=num_points, cola_relativa_cavidad=float(colas[0]),
                      cola_relativa_leaky=float(colas[1]),
                      correlaciones_decaidas=cola_ok,
                      escala_resolucion_temporal_meV=float(2 * np.pi * HBAR / tau[-1]))
    if not cola_ok:
        mensaje = (f"Las correlaciones todavía no han decaído: cola/G(0), "
                   f"cavidad={colas[0]:.3g}, leaky={colas[1]:.3g}; "
                   f"umbral={tolerancia_cola:g}. Aumenta tau_final. "
                   "Un corte aquí puede introducir oscilaciones en el espectro. "
                   "Para estudiar deliberadamente ese corte usa exigir_decaimiento=False.")
        if exigir_decaimiento:
            raise ValueError(mensaje)
        warnings.warn(mensaje, RuntimeWarning, stacklevel=2)

    u = tau / tau[-1]
    if ventana is None:
        w = np.ones_like(tau)
    elif ventana == "hann":
        w = 0.5 * (1.0 + np.cos(np.pi * u))  # w(0)=1, w(T)=0
    else:
        w = np.where(u <= 0.5, 1 - 6*u**2 + 6*u**3, 2*(1-u)**3)
    senal = correlaciones * w[:, None]
    integrada = senal * _pesos_simpson(num_points, dt)[:, None]

    def transformar_malla(inicio, fin, numero, valores=integrada, paso_t=dt):
        """Fourier de muestras temporales con pesos de Simpson, NO resolvente."""
        energias = np.linspace(inicio, fin, numero)
        frecuencias = np.array([inicio, fin]) / (2 * np.pi * HBAR)
        zoom = ZoomFFT(len(valores), frecuencias, m=numero,
                       fs=1.0 / paso_t, endpoint=True)
        espectro = zoom(valores.T, axis=-1).T.real / np.pi
        return energias, espectro

    def transformar_una(energia):
        fase = np.exp(-1j * energia * tau / HBAR)
        return (fase @ integrada).real / np.pi

    # 3. Transformar G_C(tau) y G_X(tau); refinar la malla, sin filtrar picos.
    ebase, Sbase = transformar_malla(e_min, e_max, num_energias)
    # Compara Simpson en dt y 2*dt para detectar un muestreo demasiado grueso.
    gruesa = senal[::2] * _pesos_simpson(len(senal[::2]), 2 * dt)[:, None]
    _, Sgruesa = transformar_malla(e_min, e_max, num_energias, gruesa, 2 * dt)
    escalas = np.maximum(np.max(abs(Sbase), axis=0), 1e-300)
    error_malla = np.max(abs(Sbase - Sgruesa), axis=0) / escalas
    diagnostico["diferencia_dt_vs_2dt_cavidad"] = float(error_malla[0])
    diagnostico["diferencia_dt_vs_2dt_leaky"] = float(error_malla[1])
    if np.any(error_malla[activos] > 1e-3):
        warnings.warn("El espectro cambia apreciablemente al comparar dt y 2*dt. "
                      "Aumenta num_points y comprueba convergencia.",
                      RuntimeWarning, stacklevel=2)
    energias_partes, espectros_partes = [ebase], [Sbase]
    for canal in range(2):
        if not activos[canal]:
            continue
        v = Sbase[:, canal]
        picos = np.flatnonzero((v[1:-1] > v[:-2]) & (v[1:-1] > v[2:])) + 1
        # Solo limita el refinamiento adicional; conserva TODO el espectro base.
        picos = picos[np.argsort(v[picos])[-4:]]
        for p in picos:
            if v[p] <= max(0.0, v.max() * 1e-3):
                continue
            opt = minimize_scalar(
                lambda e: -transformar_una(e)[canal],
                bounds=(ebase[p-1], ebase[p+1]), method="bounded",
                options={"xatol": 1e-10})
            if not opt.success:
                raise RuntimeError("No convergió el refinamiento de un máximo espectral.")
            es, ss = transformar_malla(opt.x - 2 * paso_base,
                                       opt.x + 2 * paso_base, 201)
            energias_partes.append(es)
            espectros_partes.append(ss)
    if xlim is not None:
        es, ss = transformar_malla(xlim[0] - EC, xlim[1] - EC, num_energias)
        energias_partes.append(es)
        espectros_partes.append(ss)
    energia, indices = np.unique(EC + np.concatenate(energias_partes), return_index=True)
    espectros = np.vstack(espectros_partes)[indices]

    # 4. Emisión externa y una normalización común para ambos canales.
    if ponderar_emision:
        espectros *= np.array([kappa_meV, gamma_meV]) / HBAR
    raw = espectros.copy()
    if not np.isfinite(raw).all():
        raise RuntimeError("El espectro contiene valores no finitos.")
    pico = float(raw.max())
    escala = max(float(np.max(abs(raw))), 1e-300)
    diagnostico["min_intensidad_relativa"] = float(raw.min() / escala)
    if raw.min() < -1e-5 * escala:
        warnings.warn("Hay intensidades negativas significativas: comprueba el corte "
                      "temporal, el paso, las tolerancias BDF y Nmax. No se han "
                      "recortado ni eliminado esos valores.", RuntimeWarning, stacklevel=2)
    factor = pico if normalizar and pico > 0 else 1.0
    espectros /= factor
    peaks = [float(energia[np.argmax(raw[:, j])]) if raw[:, j].max() > 0
             else None for j in range(2)]
    diagnostico["energia_max_cavidad_meV"] = peaks[0]
    diagnostico["energia_max_leaky_meV"] = peaks[1]

    # 5. Ambas curvas en el mismo eje.
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(energia, espectros[:, 0], lw=1.8, label="Modo de cavidad")
    ax.plot(energia, espectros[:, 1], "--", lw=1.8, label="Modos leaky / excitón")
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.set_xlabel("Energía (meV)")
    ax.set_ylabel("Intensidad normalizada" if normalizar else "Intensidad (u. arb.)")
    ax.set_title("Espectro de emisión estacionario\n"
                 + rf"$g={g_meV:g}$, $\Delta={Delta_meV:g}$, "
                 + rf"$\gamma={gamma_meV:g}$, $\kappa={kappa_meV:g}$, "
                 + rf"$P={P_meV:g}$ meV")
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()

    resultados = dict(
        energia_meV=energia, S_cavidad=espectros[:, 0],
        S_exciton=espectros[:, 1], S_leaky=espectros[:, 1],
        S_cavidad_raw=raw[:, 0], S_exciton_raw=raw[:, 1], S_leaky_raw=raw[:, 1],
        factor_normalizacion=factor, datos_rho=datos_rho,
        Nph=Nph, NX=NX, g2_0=g2, parametros=parametros,
        omegaC_meV=EC, omegaX_meV=float(omegaX_meV),
        diagnostico=diagnostico, metodo="correlaciones_BDF_Fourier_Simpson",
        tau_ps=tau, G1_cavidad_rot=correlaciones[:, 0],
        G1_exciton_rot=correlaciones[:, 1], G1_leaky_rot=correlaciones[:, 1],
        g1_cavidad_rot=normalizadas[:, 0], g1_exciton_rot=normalizadas[:, 1],
        g1_leaky_rot=normalizadas[:, 1], ventana_temporal=w,
        configuracion_temporal=dict(tau_final=float(tau_final), num_points=num_points,
                                    num_energias=num_energias, ventana=ventana,
                                    rtol=float(rtol), atol=float(atol),
                                    tolerancia_cola=float(tolerancia_cola),
                                    exigir_decaimiento=bool(exigir_decaimiento)),
        ponderar_emision=bool(ponderar_emision), normalizar=bool(normalizar),
    )
    if informar:
        print("Estado estacionario recalculado con los parámetros de esta llamada.")
        print(f"Cola relativa: cavidad={colas[0]:.3e}; leaky={colas[1]:.3e}")
        print(f"Control dt vs 2dt: cavidad={error_malla[0]:.3e}; leaky={error_malla[1]:.3e}")
        print("Espectros calculados a partir de G1(tau), con cuadratura de Simpson.")
        print(f"Traza = {diagnostico['traza']:.12f}; Nph = {Nph:.9g}; NX = {NX:.9g}")
        print(f"Residuo estacionario = {diagnostico['residuo_estacionario_ps_inv']:.3e} ps^-1")
        print(f"Población en n={Nmax}: {diagnostico['poblacion_borde']:.3e}")
        for nombre, e in zip(("cavidad", "leaky"), peaks):
            print(f"Máximo de {nombre}: {e:.9f} meV" if e is not None else f"Canal {nombre}: sin emisión.")
    if mostrar:
        plt.show()
    return fig, ax, resultados


In [ ]:
fig, ax, espectro = graficar_espectro_emision(

    # Parámetros físicos: todos en meV
    omegaX_meV=1000,
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,
    kappa_meV=0.1,
    P_meV=15,

    # Truncamiento del espacio de fotones
    Nmax=100,

    # Límites de la gráfica
    xlim=(990, 1000),
    ylim=(0, 0.01),

    # Resolución de la malla de energía
    num_energias=10001,

    # Pesos de emisión: κ para cavidad y γ para leaky
    ponderar_emision=True,

    # Un mismo máximo para normalizar ambos canales
    normalizar=True,

    figsize=(10, 6),
    mostrar=True,
    informar=True,
)

# Estado estacionario calculado automáticamente
datos_rho = espectro["datos_rho"]

# Resultados del espectro
energia = espectro["energia_meV"]
S_cavidad = espectro["S_cavidad"]
S_leaky = espectro["S_leaky"]

In [ ]:
fig, ax, espectro = graficar_espectro_emision(

    # Parámetros físicos: todos en meV
    omegaX_meV=1000,
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,
    kappa_meV=0.1,
    P_meV=15,

    # Truncamiento del espacio de fotones
    Nmax=100,

    # Límites de la gráfica
    xlim=(990, 1000),
    ylim=(0, 1),


    # Resolución de la malla de energía
    num_energias=10001,

    # Pesos de emisión: κ para cavidad y γ para leaky
    ponderar_emision=True,

    # Un mismo máximo para normalizar ambos canales
    normalizar=True,

    figsize=(10, 6),
    mostrar=True,
    informar=True,
)

# Estado estacionario calculado automáticamente
datos_rho = espectro["datos_rho"]

# Resultados del espectro
energia = espectro["energia_meV"]
S_cavidad = espectro["S_cavidad"]
S_leaky = espectro["S_leaky"]

In [ ]:
fig, ax, espectro = graficar_espectro_emision(

    # Parámetros físicos: todos en meV
    omegaX_meV=1000,
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,
    kappa_meV=5,
    P_meV=1,

    # Truncamiento del espacio de fotones
    Nmax=100,

    # Límites de la gráfica
    xlim=(980, 1015),
    ylim=(0, 1),


    # Resolución de la malla de energía
    num_energias=10001,

    # Pesos de emisión: κ para cavidad y γ para leaky
    ponderar_emision=True,

    # Un mismo máximo para normalizar ambos canales
    normalizar=True,

    figsize=(10, 6),
    mostrar=True,
    informar=True,
)

# Estado estacionario calculado automáticamente
datos_rho = espectro["datos_rho"]

# Resultados del espectro
energia = espectro["energia_meV"]
S_cavidad = espectro["S_cavidad"]
S_leaky = espectro["S_leaky"]

In [ ]:
fig, ax, espectro = graficar_espectro_emision(

    # Parámetros físicos: todos en meV
    omegaX_meV=1000,
    Delta_meV=5,
    g_meV=1,
    gamma_meV=0.1,
    kappa_meV=5,
    P_meV=15,

    # Truncamiento del espacio de fotones
    Nmax=100,

    # Límites de la gráfica
    xlim=(930, 1040),
    ylim=(0, 1),


    # Resolución de la malla de energía
    num_energias=10001,

    # Pesos de emisión: κ para cavidad y γ para leaky
    ponderar_emision=True,

    # Un mismo máximo para normalizar ambos canales
    normalizar=True,

    figsize=(10, 6),
    mostrar=True,
    informar=True,
)

# Estado estacionario calculado automáticamente
datos_rho = espectro["datos_rho"]

# Resultados del espectro
energia = espectro["energia_meV"]
S_cavidad = espectro["S_cavidad"]
S_leaky = espectro["S_leaky"]